In [0]:
%sql
--   ATLAS Marine — Notebook 2A: Silver Environmental
--   Animal Tracking and Lifecycle Analysis System

--   PURPOSE OF THIS NOTEBOOK:
--   Silver is the second layer of the ATLAS medallion architecture.
--   Its job is to take raw Bronze data and transform it into clean, validated, typed, geospatially indexed tables that are ready for analytical joins.

--   Bronze = ingredients washed and weighed (still raw)
--   Silver = ingredients chopped, seasoned and prepared (ready to cook)
--   Gold   = the finished dish (ready to serve)

--   WHAT HAPPENS AT SILVER:
--   - Cast raw strings to correct data types (DOUBLE, DATE, INTEGER)
--   - Remove sentinel values (-999, NaN, null)
--   - Filter rows to ATLAS bounding box (lat -23 to 22, lon -73 to 126)
--   - Extract year and month from filenames (SST and CC)
--   - Add H3 hexagonal geospatial index at resolution 4 (Information below - query 2)
--   - Join all environmental tables into one unified view
--   - Calculate habitat stress index (0-4) (Information below - query 3)
--
--   TABLES BUILT IN THIS NOTEBOOK:
--   (Information below - query 4)
--   1. sst_silver           — Sea surface temperature
--   2. cc_silver            — Chlorophyll concentration
--   3. cmems_silver         — Ocean physics (currents, salinity, MLD)
--   4. enso_silver          — ENSO climate index (unpivoted to long format)
--   5. microplastics_silver — Combined NCEI and AOMI microplastics
--   6. coral_bleaching_silver — Coral bleaching stress
--   7. environmental_conditions_silver — All variables joined together

In [0]:
%sql

-- BACKGROUND — H3 GEOSPATIAL INDEXING
--
-- WHAT IS H3?
-- H3 is a geospatial indexing system developed by Uber and built
-- natively into Databricks. It divides the entire surface of the
-- Earth into hexagonal cells arranged in a hierarchical grid.
-- Each cell is identified by a unique alphanumeric code called
-- an H3 index.
--
-- Hexagons are used rather than squares because:
-- - All six neighbours of a hexagon are equidistant from its centre
-- - Hexagons tessellate perfectly with no gaps or overlaps
-- - They represent geographic areas more naturally than squares
-- - Spatial joins and aggregations are more accurate
--
-- WHAT IS RESOLUTION?
-- H3 has 16 resolution levels from 0 (very large) to 15 (very small).
-- Higher resolution = smaller cells = more precise location
-- Lower resolution = larger cells = broader geographic area
--
-- H3 RESOLUTION REFERENCE TABLE:
-- Resolution 0  — ~4,250,000 km² (continent sized)
-- Resolution 1  — ~600,000 km²
-- Resolution 2  — ~86,000 km²
-- Resolution 3  — ~12,000 km²
-- Resolution 4  — ~1,770 km²   ← ATLAS primary join resolution
-- Resolution 5  — ~252 km²     ← ATLAS secondary resolution
-- Resolution 6  — ~36 km²      ← ATLAS fine-grain resolution
-- Resolution 7  — ~5 km²
-- Resolution 8  — ~0.7 km²
--
-- WHY RESOLUTION 4 FOR ATLAS?
-- The choice of resolution 4 is driven by our coarsest data source.
-- Our environmental datasets have different spatial resolutions:
--
-- SST  (NOAA OI SST V2)    — 0.25 degree (~25km grid cells)
-- CC   (NOAA MODIS Aqua)   — 1 degree    (~100km grid cells)
-- CMEMS (GLORYS12V1)       — 0.083 degree (~8km grid cells)
--
-- At resolution 4 each H3 cell covers approximately 1,770 km².
-- This is large enough that multiple SST grid cells (25km)
-- and at least one CC grid cell (100km) fall within each H3 cell.
-- This means every H3 cell has both SST and CC data — preventing
-- large numbers of null values in the joined Silver table.
--
-- If we used a finer resolution (e.g. resolution 7 at ~5km):
-- - SST cells at 25km would not always overlap with H3 cells at 5km
-- - CC cells at 100km would span many H3 cells producing nulls
-- - Many H3 cells would have SST but no CC or vice versa
--
-- If we used a coarser resolution (e.g. resolution 2 at ~86,000km²):
-- - Cells would be too large — we would lose geographic precision
-- - Different species localities (Raja Ampat vs Komodo) would
--   be in the same H3 cell and we could not distinguish them
--
-- Resolution 4 at ~1,770 km² is the sweet spot — large enough
-- for all environmental data sources to have coverage in every
-- cell, small enough to distinguish different geographic localities
-- within our study region.
--
-- WHY RESOLUTION 5 AS WELL?
-- Resolution 5 cells (~252 km²) are used for joining species
-- tracking data to environmental conditions. Tracking positions
-- are precise GPS points — they need a finer resolution cell
-- to preserve geographic meaning at the scale of individual
-- animal movements.
--
-- Resolution 6 (~36 km²) is calculated and stored for future
-- fine-grain analysis — for example identifying specific reef
-- structures within Raja Ampat where manta rays aggregate.
--
-- HOW H3 IS USED IN ATLAS:
-- Step 1: Every environmental measurement (SST, CC, CMEMS etc)
--         is assigned an H3 index using its latitude and longitude
--         h3_longlatash3(longitude, latitude, resolution)
--
-- Step 2: Every species tracking observation is assigned the
--         same H3 index using its GPS coordinates
--
-- Step 3: Tables are joined on the H3 index — all measurements
--         in the same H3 cell are treated as spatially co-located
--         even if their exact coordinates differ slightly
--
-- This solves the fundamental problem of joining datasets with
-- different spatial resolutions without requiring exact coordinate
-- matching — which would produce very few matches given the
-- different grid sizes of our seven data sources.


In [0]:
%sql

-- BACKGROUND — HABITAT STRESS INDEX
--
-- WHAT IS THE HABITAT STRESS INDEX?
-- The Habitat Stress Index (HSI) is a composite score developed
-- for ATLAS that combines four independent environmental stressors
-- into a single number ranging from 0 to 4.
--
-- It answers the question: how many simultaneous environmental
-- threats is this location experiencing right now?
--
-- A score of 0 means none of the four stressors are present —
-- conditions are within historically normal ranges.
-- A score of 4 means all four stressors are present simultaneously
-- — the most severely degraded habitat conditions in the dataset.
--
-- WHY A COMPOSITE INDEX?
-- Marine animals do not experience stressors in isolation.
-- A manta ray in Raja Ampat does not just face thermal stress
-- or just food stress — it faces all of them simultaneously.
-- Individual variables like SST or chlorophyll tell part of the
-- story. The HSI tells the whole story in a single number that
-- is immediately interpretable by scientists, policy makers and
-- conservationists alike.
--
-- This approach is consistent with established multi-stressor
-- frameworks in marine conservation science — the same principle
-- used by the Ocean Health Index and similar platforms.
--
-- THE FOUR STRESSORS AND THEIR THRESHOLDS:
--
-- STRESSOR 1 — THERMAL STRESS
-- Threshold: SST > 30°C
-- Why 30°C? This is the widely accepted upper thermal tolerance
-- limit for most Indo-Pacific coral reef ecosystems. Above 30°C
-- coral bleaching stress accelerates significantly. For manta rays
-- and whale sharks — both filter feeders — temperatures above 30°C
-- are associated with reduced plankton productivity and altered
-- migration patterns. The 30°C threshold is consistent with NOAA
-- Coral Reef Watch bleaching alert methodology.
--
-- STRESSOR 2 — FOOD STRESS
-- Threshold: Chlorophyll < 0.1 mg/m³
-- Why 0.1 mg/m³? Values below 0.1 mg/m³ indicate oligotrophic
-- (nutrient-poor) ocean conditions — sometimes called ocean deserts.
-- Reef manta rays and whale sharks are obligate filter feeders
-- that require dense zooplankton concentrations to meet their
-- energy requirements. These zooplankton populations depend on
-- phytoplankton measured by chlorophyll. Below 0.1 mg/m³ the
-- food chain supporting filter feeders is severely depleted.
--
-- STRESSOR 3 — POLLUTION STRESS
-- Threshold: Microplastic concentration > 1.0 pieces/m³
-- Why 1.0 pieces/m³? This threshold represents elevated
-- microplastic contamination above typical background ocean levels.
-- Filter feeders like manta rays and whale sharks are particularly
-- vulnerable to microplastic ingestion because they filter thousands
-- of cubic metres of water per day. Concentrations above 1.0
-- pieces/m³ have been associated with measurable microplastic
-- loads in the digestive systems of filter-feeding megafauna
-- in published marine biology literature.
--
-- STRESSOR 4 — BLEACHING STRESS
-- Threshold: Coral Reef Watch Bleaching Alert Area (BAA) >= 2
-- Why BAA >= 2? A BAA of 2 corresponds to a Warning level —
-- the point at which significant coral bleaching is expected.
-- This threshold is the internationally recognised early warning
-- level used by coral reef managers and conservation organisations
-- worldwide. For species like reef manta rays whose feeding habitat
-- depends on healthy coral reef ecosystems a BAA of 2 or above
-- signals that habitat quality is being actively degraded.
--
-- WHY BINARY SCORING (0 or 1 per stressor)?
-- Each stressor contributes exactly 1 point when its threshold
-- is exceeded — regardless of how far above the threshold it is.
-- This is intentional. A binary approach:
-- - Makes the index interpretable — each point means one more
--   stressor is present, not a weighted combination of magnitudes
-- - Avoids the need to normalise variables with very different
--   units (°C, mg/m³, pieces/m³, categorical scale)
-- - Treats each stressor as equally important — consistent with
--   the precautionary principle in conservation science
-- - Allows straightforward comparison across time periods,
--   species and geographic locations
--
-- KNOWN LIMITATIONS:
-- The binary approach does not capture severity within a stressor.
-- An SST of 30.1°C and 35°C both score 1 — the severity is lost.
-- This is a deliberate simplification appropriate for a
-- multi-stressor summary index. Severity analysis can be done
-- separately on the individual variables.
--
-- The microplastics data is sparse — many H3 cells have no
-- microplastics measurements and will show a null rather than
-- a score. This means the pollution component of the HSI
-- underestimates the true pollution burden in data-sparse areas.
-- This limitation is documented in the ATLAS methodology.
--
-- HOW TO INTERPRET THE HSI IN ATLAS ANALYSIS:
-- HSI 0 — No stress — baseline conditions
-- HSI 1 — Single stressor — one threat present
-- HSI 2 — Dual stress — two threats simultaneously
-- HSI 3 — Triple stress — three threats simultaneously
-- HSI 4 — Critical stress — all four threats simultaneously
--
-- H3 cells scoring 4 with high species presence are the highest
-- priority conservation concern in the ATLAS dataset — animals
-- are present in maximally degraded habitat. H3 cells scoring 4
-- with low or zero species presence may indicate habitat that
-- has already been abandoned by the species.
--
-- THE 2026 EL NINO CONNECTION:
-- During the 2026 El Nino event — one of the strongest ever
-- recorded — SST anomalies of +2°C above the 30-year baseline
-- have been observed across the ATLAS study region using
-- Google Earth Engine satellite data. This means many H3 cells
-- that previously scored 0 or 1 on thermal stress are now
-- scoring 1 on thermal stress alone — and combined with
-- ongoing bleaching stress may be scoring 2 or higher across
-- previously pristine reef habitat.


In [0]:
%sql

-- ─────────────────────────────────────────────────────────────
-- TABLE 1 — sst_silver
-- Sea Surface Temperature
-- Source: NOAA Optimum Interpolation SST Version 2 (OI SST V2)
-- Coverage: 1993 to 2026 — 33 years
-- Spatial resolution: 0.25 degree (~25km grid cells)
-- ─────────────────────────────────────────────────────────────
-- WHAT IT IS:
-- Sea surface temperature measures how warm the top layer of the
-- ocean is at any given location and time. It is recorded in
-- degrees Celsius and is one of the most fundamental measurements
-- in ocean science.
--
-- NOAA combines satellite observations from multiple sensors with
-- ship and buoy measurements using an optimal interpolation
-- algorithm to produce a complete daily global grid with no gaps.
-- ATLAS uses monthly averages derived from these daily values.
--
-- WHY IT IS IN ATLAS:
-- SST is the single most important environmental variable for
-- marine animals in tropical waters. It directly determines:
-- - Whether coral reefs bleach (above ~28-30°C for sustained periods)
-- - Where phytoplankton blooms occur (drives food availability)
-- - Which migration routes animals can use (thermal barriers)
-- - Whether habitat is physiologically suitable for a species
--
-- For reef manta rays in Raja Ampat — SST determines whether
-- their coral reef habitat is bleaching and whether the thermal
-- conditions support the zooplankton they feed on.
-- For whale sharks in the Gulf of Mexico — SST influences the
-- timing and location of seasonal aggregations.
-- For green turtles in the Indian Ocean — SST affects nesting
-- beach selection and migration corridor viability.
--
-- THE KEY ATLAS FINDING IS AN SST FINDING:
-- La Nina average SST (29.47°C) now exceeds El Nino average SST
-- (28.80°C) in reef manta ray habitat in Raja Ampat — the
-- opposite of what ENSO climate science predicts. This finding
-- — that long-term ocean warming is overriding the expected
-- La Nina cooling signal — is only visible because ATLAS holds
-- 33 years of monthly SST data with ENSO phase context.
--
-- WHY 1993?
-- The NOAA OI SST V2 product is available from 1981. ATLAS starts
-- at 1993 because this is when the CMEMS ocean reanalysis begins —
-- aligning all environmental datasets to the same start date
-- ensures consistent multi-variable analysis across the full
-- study period.
--
-- ─────────────────────────────────────────────────────────────
-- TABLE 2 — cc_silver
-- Chlorophyll Concentration
-- Source: NASA MODIS Aqua Ocean Color via NOAA CoastWatch ERDDAP
-- Coverage: 2002 to 2026 — 24 years
-- Spatial resolution: 1 degree (~100km grid cells)
-- ─────────────────────────────────────────────────────────────
-- WHAT IT IS:
-- Chlorophyll concentration measures the abundance of
-- phytoplankton — microscopic plant-like organisms — in the
-- surface ocean. It is measured in milligrams per cubic metre
-- (mg/m³) using satellite ocean colour sensors that detect
-- the green wavelengths absorbed by chlorophyll pigment.
--
-- High chlorophyll = lots of phytoplankton = productive ocean
-- Low chlorophyll = few phytoplankton = food-poor ocean desert
--
-- The NASA MODIS Aqua sensor was launched in 2002 — this is
-- why CC data starts in 2002 rather than 1993. There is no
-- equivalent quality global ocean colour dataset before 2002.
-- July 2002 is the first reliable month of data.
--
-- WHY IT IS IN ATLAS:
-- Chlorophyll is the foundation of the marine food web. Every
-- animal in the ocean ultimately depends on phytoplankton.
-- For filter feeders like reef manta rays and whale sharks —
-- which eat zooplankton that eat phytoplankton — chlorophyll
-- concentration is a direct proxy for food availability.
--
-- When chlorophyll is low (below ~0.1 mg/m³) the food chain
-- that supports filter feeding megafauna collapses. The animals
-- must either move to more productive waters or face starvation.
-- This is why low chlorophyll contributes to the habitat stress
-- index — it signals food-poor conditions directly.
--
-- CC ALSO VALIDATES THE ENSO FINDING:
-- ATLAS found that El Nino years show HIGHER chlorophyll
-- (0.426 mg/m³) than La Nina years (0.367 mg/m³) in Raja Ampat
-- — also counterintuitive. This suggests filter feeders may
-- actually experience better feeding conditions during El Nino
-- than La Nina — the opposite of what we would expect if ENSO
-- were behaving normally.
--
-- KNOWN LIMITATION:
-- May 2024 is missing from the NOAA source data — a known gap
-- in the satellite record documented in the ATLAS data quality
-- framework.
--
-- ─────────────────────────────────────────────────────────────
-- TABLE 3 — cmems_silver
-- Ocean Physics Reanalysis
-- Source: Copernicus Marine Environment Monitoring Service
--         GLORYS12V1 — GLOBAL_MULTIYEAR_PHY_001_030
-- Coverage: 1993 to 2026 — 34 years
-- Spatial resolution: 0.083 degree (~8km grid cells)
-- ─────────────────────────────────────────────────────────────
-- WHAT IT IS:
-- CMEMS (Copernicus Marine Environment Monitoring Service) is
-- the EU ocean monitoring programme. The GLORYS12V1 reanalysis
-- product combines satellite observations with ocean physics
-- models to reconstruct the complete three-dimensional state
-- of the global ocean from 1993 to present.
--
-- A reanalysis means scientists took all available observations
-- from 1993 onwards and used an ocean circulation model to fill
-- the gaps — producing a physically consistent reconstruction
-- of what the ocean was doing everywhere at every point in time.
--
-- Variables in ATLAS:
-- current_east (uo)    — eastward current velocity (m/s)
-- current_north (vo)   — northward current velocity (m/s)
-- current_speed        — total current speed derived from uo and vo
-- sea_level_anomaly    — how much higher or lower than average (m)
-- mixed_layer_depth    — depth of the well-mixed surface layer (m)
-- salinity             — salt content of the water (PSU)
-- ocean_temperature    — cross-check against NOAA SST (°C)
--
-- WHY IT IS IN ATLAS:
-- SST and chlorophyll tell us about surface conditions.
-- CMEMS tells us about the physics of the ocean itself —
-- how water is moving, how salty it is, how deep the productive
-- surface layer goes.
--
-- Current speed and direction: Marine animals use ocean currents
-- for energy-efficient migration. Manta rays ride current
-- boundaries. Whale sharks follow upwelling currents that
-- concentrate their prey. Understanding currents helps explain
-- why animals are in specific locations.
--
-- Mixed layer depth: A shallow mixed layer concentrates nutrients
-- near the surface where light penetrates — creating productive
-- feeding conditions. A deep mixed layer disperses nutrients
-- reducing surface productivity. This directly affects food
-- availability for filter feeders.
--
-- Salinity: River outflows create low-salinity plumes in coastal
-- waters. These plumes are known aggregation sites for whale
-- sharks and manta rays — possibly because they concentrate
-- organic matter and zooplankton at the freshwater-saltwater
-- boundary.
--
-- Sea level anomaly: Elevated sea level indicates warm water
-- accumulation — an early warning signal for thermal stress
-- before it shows in SST measurements.
--
-- ─────────────────────────────────────────────────────────────
-- TABLE 4 — enso_silver
-- ENSO Climate Index
-- Source: NOAA Physical Sciences Laboratory — MEI.v2
-- Coverage: 1979 to 2026 — 46 years
-- Format: Monthly global index — no spatial dimension
-- ─────────────────────────────────────────────────────────────
-- WHAT IT IS:
-- ENSO stands for El Nino Southern Oscillation — the most
-- powerful natural climate cycle on Earth. It describes a
-- recurring pattern of ocean temperature and atmospheric
-- pressure variation across the tropical Pacific Ocean that
-- operates on an irregular 2-7 year cycle.
--
-- The Multivariate ENSO Index Version 2 (MEI.v2) is the
-- scientific standard for measuring ENSO. Unlike simpler
-- indices that only measure Pacific SST, MEI.v2 combines
-- five variables: sea surface temperature, sea level pressure,
-- surface winds (east-west and north-south) and outgoing
-- longwave radiation (a proxy for cloud cover).
--
-- ENSO has three phases:
-- El Nino  — MEI.v2 > +0.5 — warm central Pacific
-- La Nina  — MEI.v2 < -0.5 — cool central Pacific
-- Neutral  — between -0.5 and +0.5
--
-- WHY IT IS IN ATLAS:
-- ENSO is essential context for interpreting every other
-- environmental variable. Without knowing the ENSO phase
-- we cannot tell whether an unusual SST reading in Raja Ampat
-- is a long-term warming signal or a temporary El Nino effect.
--
-- ENSO affects the entire global ocean through atmospheric
-- teleconnections — changes in trade winds, jet streams and
-- ocean heat transport that alter temperature and productivity
-- patterns far beyond the Pacific. The Indo-Pacific where our
-- manta rays live, the Indian Ocean where our green turtles
-- migrate and the Gulf of Mexico where our whale sharks and
-- sperm whales live are all significantly affected by ENSO.
--
-- THE ENSO FINDING:
-- ATLAS analysis showed La Nina average SST in Raja Ampat
-- (29.47°C) exceeds El Nino average SST (28.80°C) — the
-- opposite of what ENSO predicts. La Nina is supposed to cool
-- the Indo-Pacific. It no longer does — at least not in our
-- study area. This is the central finding of ATLAS Phase 4.
--
-- UNPIVOTING EXPLAINED:
-- The MEI.v2 file arrives in wide format — one row per year
-- with 12 monthly values as columns. ATLAS Silver unpivots
-- this to long format — one row per year per month — so it
-- can be joined to the monthly SST and CC data on year and
-- month columns. ENSO has no spatial dimension — every location
-- in the world in the same month gets the same ENSO value.
--
-- ─────────────────────────────────────────────────────────────
-- TABLE 5 — microplastics_silver
-- Ocean Microplastics Concentration
-- Source 1: NOAA NCEI — Atlantic and Caribbean — 2010 to 2015
-- Source 2: AOMI — Indo-Pacific — 1986 to 2024
-- Combined coverage: Atlantic, Indian Ocean, Pacific
-- ─────────────────────────────────────────────────────────────
-- WHAT IT IS:
-- Microplastics are plastic particles smaller than 5mm that
-- have entered the ocean through river discharge, coastal
-- littering, degradation of larger plastic items, synthetic
-- textile fibres from laundry and industrial pellet spills.
-- They are measured as pieces per cubic metre (pieces/m³).
--
-- Two datasets are combined in ATLAS:
--
-- NOAA NCEI (National Centers for Environmental Information):
-- Compiled microplastic measurements from dedicated research
-- cruises in the Atlantic Ocean and Caribbean Sea. 471 sampling
-- stations covering 2010 to 2015. The most rigorous quality
-- controlled microplastics dataset available for the Atlantic.
--
-- AOMI (Atmosphere and Ocean Research Institute, Japan):
-- A global database of ocean microplastic measurements with
-- particular coverage of the Indo-Pacific — the Indian Ocean,
-- Indonesian waters and Northwestern Pacific. 1,682 sampling
-- stations covering 1986 to 2024. Fills the critical gap left
-- by the Atlantic-focused NCEI dataset.
--
-- WHY IT IS IN ATLAS:
-- Microplastics represent a chronic pollution threat to all
-- four ATLAS species — but especially to filter feeders.
--
-- Reef manta rays filter thousands of cubic metres of water
-- per day to capture zooplankton. At concentrations above
-- 1.0 pieces/m³ they unavoidably ingest significant quantities
-- of microplastic particles. Studies on manta ray populations
-- in the Indo-Pacific have found microplastics in stomach
-- content analysis — the particles accumulate in digestive
-- tissue, cause inflammation and reduce feeding efficiency.
--
-- Whale sharks face identical risks — they are the largest
-- filter feeders on Earth and their Gulf of Mexico feeding
-- grounds are in a region with significant plastic pollution
-- from the Mississippi River drainage basin.
--
-- Green turtles ingest microplastics both through direct
-- consumption (mistaking particles for food) and through
-- eating contaminated prey. The Indian Ocean, while less
-- polluted than the Atlantic, has documented microplastic
-- accumulation zones in the Chagos region.
--
-- WHY TWO SOURCES COMBINED?
-- No single global microplastics dataset covers all three
-- ATLAS ocean basins at adequate sampling density. NCEI covers
-- the Gulf of Mexico and Caribbean (whale shark and sperm whale
-- habitat) while AOMI covers the Indo-Pacific (manta ray habitat)
-- and Indian Ocean (green turtle habitat). Together they provide
-- the most complete microplastics picture available for ATLAS.
--
-- ─────────────────────────────────────────────────────────────
-- TABLE 6 — coral_bleaching_silver
-- Coral Reef Thermal Stress
-- Source: NOAA Coral Reef Watch 5km Satellite Product Suite v3.1
-- Coverage: 1985 to 2026 — 41 years
-- Spatial resolution: 5km (0.05 degree)
-- ─────────────────────────────────────────────────────────────
-- WHAT IT IS:
-- The NOAA Coral Reef Watch (CRW) programme produces daily
-- satellite-derived monitoring products specifically designed
-- to detect and predict coral bleaching events.
--
-- Two variables are used in ATLAS:
--
-- Degree Heating Week (CRW_DHW):
-- Measures accumulated heat stress above the coral bleaching
-- threshold over the most recent 12-week rolling period.
-- Units are degree-weeks (°C-weeks).
-- DHW 0    = no thermal stress
-- DHW >= 4 = bleaching likely
-- DHW >= 8 = severe bleaching and coral mortality likely
-- DHW >= 16 = catastrophic mortality likely
--
-- Bleaching Alert Area (CRW_BAA):
-- A categorical 0-5 alert scale derived from DHW values:
-- 0 = No stress
-- 1 = Watch
-- 2 = Warning — bleaching likely (ATLAS HSI threshold)
-- 3 = Alert Level 1
-- 4 = Alert Level 2
-- 5 = Alert Level 2+
--
-- ATLAS downloaded annual snapshots using stride=365 — one
-- observation per year in April which captures the accumulated
-- heat stress from the preceding 12 weeks — the peak bleaching
-- season in the northern hemisphere tropics.
--
-- WHY IT IS IN ATLAS:
-- Three of the four ATLAS species live in or depend on coral
-- reef ecosystems. Coral bleaching is the most direct and
-- measurable consequence of ocean warming for these species.
--
-- Reef manta rays feed on zooplankton that concentrate around
-- healthy reef structures. When coral bleaches and dies the
-- reef ecosystem collapses — fish populations decline,
-- zooplankton patterns shift and manta ray feeding grounds
-- are destroyed. There is no substitute habitat for reef
-- manta rays — they cannot simply move to a different ecosystem.
--
-- Green turtles feed on seagrass and algae in shallow reef
-- areas. Bleaching events directly degrade these feeding
-- habitats and can displace turtles from historical foraging
-- grounds they have used for generations.
--
-- Whale sharks aggregate in areas of high productivity.
-- Bleaching events suppress the chlorophyll productivity
-- that whale sharks depend on — adding a third mechanism
-- (alongside SST and chlorophyll) connecting ENSO phase to
-- whale shark presence in the Gulf of Mexico.
--
-- THE 2026 EL NINO CONNECTION:
-- The 2026 El Nino — one of the strongest ever recorded —
-- is producing SST anomalies of +2°C above the 30-year
-- baseline across the ATLAS study region. This is driving
-- DHW values well above the bleaching threshold in Raja Ampat
-- manta ray habitat — visible in real time through the ATLAS
-- Google Earth Engine satellite monitoring layer.
--
-- ─────────────────────────────────────────────────────────────
-- TABLE 7 — environmental_conditions_silver
-- Master Environmental Join Table
-- Source: All six Silver tables joined together
-- Coverage: 1993 to 2026 — driven by SST as primary table
-- ─────────────────────────────────────────────────────────────
-- WHAT IT IS:
-- This is the most important table in the entire ATLAS pipeline.
-- It joins all six environmental Silver tables into a single
-- unified view keyed by H3 grid cell, year and month.
--
-- Every row in this table represents one H3 grid cell in one
-- month of one year — with every available environmental
-- measurement attached to that cell and time period.
--
-- It is the table that answers questions like:
-- "What was the SST, chlorophyll, ENSO phase, microplastic
-- concentration, ocean current speed and bleaching stress in
-- the reef manta ray habitat of Raja Ampat in January 2016
-- during the peak of the very strong El Nino?"
--
-- Without this table answering that question would require
-- six separate queries across six separate tables and manual
-- coordination of results. With this table it is one SELECT.
--
-- WHY SST IS THE PRIMARY TABLE:
-- SST has the most complete coverage of all datasets:
-- - 33 years of monthly data (1993-2026)
-- - Consistent global grid with no gaps
-- - Covers our entire bounding box
--
-- Every other dataset joins to SST using LEFT JOIN — meaning
-- all SST rows are preserved even where other data is missing.
-- This ensures we never lose temporal or spatial coverage
-- because of gaps in other datasets.
--
-- WHAT NULL VALUES MEAN IN THIS TABLE:
-- NULL chlorophyll before 2002 — MODIS Aqua not yet launched
-- NULL chlorophyll for May 2024 — known NOAA source gap
-- NULL CMEMS for some cells — grid resolution mismatch
-- NULL microplastics for most cells — sparse sampling data
-- NULL bleaching for pre-1985 — CRW data starts 1985
-- All nulls are expected, documented and handled in Gold
--
-- THE HABITAT STRESS INDEX IN THIS TABLE:
-- The HSI (0-4) is calculated here at Silver rather than Gold
-- because it is derived purely from environmental variables —
-- no species data is needed. Calculating it at Silver means
-- it is available for any downstream analysis in Gold without
-- recalculating each time.
--
-- THIS TABLE IS THE FOUNDATION OF EVERYTHING:
-- Gold views query this table
-- Species tracking joins to this table
-- AI agents query this table
-- Tableau dashboards connect to Gold views built on this table
-- The smoking gun ENSO finding was first discovered in this table


In [0]:
%sql

-- CHECK
-- Confirm all Bronze tables exist before building Silver
-- RUN THIS FIRST!! — do not proceed if any table shows zero

 
SELECT 'bronze_sst_master' AS table_name, count(*) AS rows FROM mcphersonsharyn_bronze.bronze_sst_master
UNION ALL
SELECT 'bronze_cc_master', count(*) FROM mcphersonsharyn_bronze.bronze_cc_master
UNION ALL
SELECT 'bronze_cmems_raw', count(*) FROM mcphersonsharyn_bronze.bronze_cmems_raw
UNION ALL
SELECT 'bronze_enso_raw', count(*) FROM mcphersonsharyn_bronze.bronze_enso_raw
UNION ALL
SELECT 'bronze_microplastics_raw', count(*) FROM mcphersonsharyn_bronze.bronze_microplastics_raw
UNION ALL
SELECT 'bronze_microplastics_aomi_raw', count(*) FROM mcphersonsharyn_bronze.bronze_microplastics_aomi_raw
UNION ALL
SELECT 'bronze_coral_bleaching_raw', count(*) FROM mcphersonsharyn_bronze.bronze_coral_bleaching_raw
ORDER BY table_name;

-- All tables should show non-zero row counts
-- If any show zero re-run the relevant Bronze section first

In [0]:
%sql
SELECT
  year,
  latitude,
  longitude,
  sst_raw
FROM mcphersonsharyn_bronze.bronze_sst_master
LIMIT 5;
-- Confirms year column now exists in Bronze SST
-- We can use year directly in Silver — no filename parsing needed
-- latitude, longitude and sst_raw are the other three columns

In [0]:
%sql
SELECT DISTINCT year
FROM mcphersonsharyn_bronze.bronze_sst_master
WHERE year = 2018
LIMIT 5;
-- Checking if month information exists anywhere
-- If only year is available we need another approach for month
-- Run this to see all unique columns

In [0]:
%sql
SELECT
  year,
  month,
  latitude,
  longitude,
  sst_raw
FROM mcphersonsharyn_bronze.bronze_sst_master
LIMIT 5;
-- Confirms year and month columns exist in Bronze SST
-- Ready to build SST Silver without filename parsing

In [0]:
%sql
CREATE OR REPLACE TABLE mcphersonsharyn_silver.sst_silver
USING DELTA AS
SELECT
  CAST(latitude AS DOUBLE) AS latitude,
  CAST(longitude AS DOUBLE) AS longitude,
  CAST(sst_raw AS DOUBLE) AS sea_surface_temperature,
  year,
  month,
  h3_longlatash3(
    CAST(longitude AS DOUBLE),
    CAST(latitude AS DOUBLE),
    4
  ) AS h3_res4,
  h3_longlatash3(
    CAST(longitude AS DOUBLE),
    CAST(latitude AS DOUBLE),
    5
  ) AS h3_res5
FROM mcphersonsharyn_bronze.bronze_sst_master
WHERE CAST(sst_raw AS DOUBLE) != -999
AND CAST(sst_raw AS DOUBLE) IS NOT NULL
AND CAST(sst_raw AS DOUBLE) BETWEEN -2 AND 40
AND CAST(latitude AS DOUBLE) BETWEEN -23 AND 22
AND CAST(longitude AS DOUBLE) BETWEEN -73 AND 126;
-- Creates SST Silver with year and month directly from Bronze
-- Removes -999 sentinel values and physically impossible SST values
-- Filters to ATLAS bounding box
-- Adds H3 index at resolution 4 and 5 for spatial joining
-- This will take a few minutes given 3 million rows

In [0]:
%python
from pyspark.sql.functions import col, expr

df = spark.table("mcphersonsharyn_bronze.bronze_sst_master") \
    .filter(col("sst_raw") != -999) \
    .filter(col("sst_raw").isNotNull()) \
    .filter((col("sst_raw") >= -2) & (col("sst_raw") <= 40)) \
    .filter((col("latitude") >= -23) & (col("latitude") <= 22)) \
    .filter((col("longitude") >= -73) & (col("longitude") <= 126)) \
    .withColumn("sea_surface_temperature", col("sst_raw").cast("double")) \
    .withColumn("h3_res4", expr("h3_longlatash3(longitude, latitude, 4)")) \
    .withColumn("h3_res5", expr("h3_longlatash3(longitude, latitude, 5)")) \
    .select("latitude", "longitude", "sea_surface_temperature",
            "year", "month", "h3_res4", "h3_res5")

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("mcphersonsharyn_silver.sst_silver")

print(f"Rows written: {df.count()}")
# expr() calls Databricks built-in H3 functions from Python
# h3_longlatash3(longitude, latitude, resolution) returns H3 index
# All Silver cleaning rules applied before writing

In [0]:
%python
from pyspark.sql.functions import col

# Check row counts at each filter stage
total = spark.table("mcphersonsharyn_bronze.bronze_sst_master").count()
print(f"Total Bronze rows: {total}")

after_999 = spark.table("mcphersonsharyn_bronze.bronze_sst_master") \
    .filter(col("sst_raw") != -999).count()
print(f"After removing -999: {after_999}")

after_null = spark.table("mcphersonsharyn_bronze.bronze_sst_master") \
    .filter(col("sst_raw") != -999) \
    .filter(col("sst_raw").isNotNull()).count()
print(f"After removing nulls: {after_null}")

after_range = spark.table("mcphersonsharyn_bronze.bronze_sst_master") \
    .filter(col("sst_raw") != -999) \
    .filter(col("sst_raw").isNotNull()) \
    .filter((col("sst_raw") >= -2) & (col("sst_raw") <= 40)).count()
print(f"After SST range filter: {after_range}")
# Shows which filter is removing too many rows
# The culprit filter will show a dramatic drop in count

In [0]:
%python
from pyspark.sql.functions import col

# Check the latitude and longitude ranges in Bronze
spark.table("mcphersonsharyn_bronze.bronze_sst_master") \
    .selectExpr(
        "min(latitude) as min_lat",
        "max(latitude) as max_lat", 
        "min(longitude) as min_lon",
        "max(longitude) as max_lon"
    ).show()
# Shows actual coordinate ranges in Bronze SST
# If longitude shows values outside -73 to 126 that explains the drop
# The SST data may cover global coordinates not just our bounding box

In [0]:
%python
from pyspark.sql.functions import col

# Count rows within bounding box
within_bbox = spark.table("mcphersonsharyn_bronze.bronze_sst_master") \
    .filter(col("sst_raw") != -999) \
    .filter((col("latitude") >= -23) & (col("latitude") <= 22)) \
    .filter((col("longitude") >= -73) & (col("longitude") <= 126)) \
    .count()
print(f"Rows within bounding box: {within_bbox}")

# Check longitude distribution
spark.table("mcphersonsharyn_bronze.bronze_sst_master") \
    .filter(col("sst_raw") != -999) \
    .filter((col("latitude") >= -23) & (col("latitude") <= 22)) \
    .selectExpr(
        "min(longitude) as min_lon",
        "max(longitude) as max_lon",
        "count(*) as total"
    ).show()
# Checks how many rows fall within our longitude range
# If very few rows are in -73 to 126 the SST data may use different coordinates

In [0]:
%python
from pyspark.sql.functions import col

# Check longitude distribution in bands
spark.table("mcphersonsharyn_bronze.bronze_sst_master") \
    .filter(col("sst_raw") != -999) \
    .filter((col("latitude") >= -23) & (col("latitude") <= 22)) \
    .selectExpr(
        "CASE WHEN longitude BETWEEN -180 AND -90 THEN '-180 to -90'" +
        "     WHEN longitude BETWEEN -90 AND 0 THEN '-90 to 0'" +
        "     WHEN longitude BETWEEN 0 AND 90 THEN '0 to 90'" +
        "     WHEN longitude BETWEEN 90 AND 180 THEN '90 to 180'" +
        "     ELSE 'other' END as lon_band",
        "longitude"
    ).groupBy("lon_band").count().orderBy("lon_band").show()
# Shows how longitude values are distributed
# This tells us where the bulk of the data sits
# Our bounding box may need adjusting

In [0]:
%python
from pyspark.sql.functions import col

# Show actual longitude values
spark.table("mcphersonsharyn_bronze.bronze_sst_master") \
    .filter(col("sst_raw") != -999) \
    .filter((col("latitude") >= -23) & (col("latitude") <= 22)) \
    .select("latitude", "longitude", "sst_raw", "year", "month") \
    .orderBy("longitude") \
    .show(10)
# Shows actual longitude values in order
# Looking for the pattern — are they 0-360 format or -180 to 180?

In [0]:
%python
from pyspark.sql.functions import col, expr

df = spark.table("mcphersonsharyn_bronze.bronze_sst_master") \
    .filter(col("sst_raw") != -999) \
    .filter(col("sst_raw").isNotNull()) \
    .filter((col("sst_raw") >= -2) & (col("sst_raw") <= 40)) \
    .filter((col("latitude") >= -23) & (col("latitude") <= 22)) \
    .withColumn("sea_surface_temperature", col("sst_raw").cast("double")) \
    .withColumn("h3_res4", expr("h3_longlatash3(longitude, latitude, 4)")) \
    .withColumn("h3_res5", expr("h3_longlatash3(longitude, latitude, 5)")) \
    .select("latitude", "longitude", "sea_surface_temperature",
            "year", "month", "h3_res4", "h3_res5")

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("mcphersonsharyn_silver.sst_silver")

print(f"Rows written: {df.count()}")
# Removed longitude bounding box filter
# SST files cover full globe — longitude filter too restrictive
# Latitude filter -23 to 22 still applied — constrains tropical band
# Species join in Gold will naturally constrain to study region
# H3 indexing handles spatial filtering at analysis stage

In [0]:
%sql
SELECT
  'sst_silver' AS table_name,
  count(*) AS total_rows,
  count(DISTINCT year) AS distinct_years,
  min(year) AS earliest_year,
  max(year) AS latest_year,
  count(DISTINCT month) AS distinct_months,
  round(min(sea_surface_temperature), 2) AS min_sst,
  round(max(sea_surface_temperature), 2) AS max_sst,
  round(avg(sea_surface_temperature), 2) AS avg_sst,
  count(CASE WHEN h3_res4 IS NULL THEN 1 END) AS null_h3
FROM mcphersonsharyn_silver.sst_silver;
-- Confirms SST Silver built correctly
-- min_sst should be above -2, max_sst below 40
-- null_h3 should be 0
-- distinct_years should be 34, distinct_months 12


In [0]:
%python
from pyspark.sql.functions import col, expr

df = spark.table("mcphersonsharyn_bronze.bronze_cc_master") \
    .filter(col("cc_raw") > 0) \
    .filter(col("cc_raw").isNotNull()) \
    .filter(col("cc_raw") < 100) \
    .filter((col("latitude") >= -23) & (col("latitude") <= 22)) \
    .withColumn("chlorophyll_concentration", col("cc_raw").cast("double")) \
    .withColumn("h3_res4", expr("h3_longlatash3(longitude, latitude, 4)")) \
    .select("latitude", "longitude", "chlorophyll_concentration",
            "year", "month", "h3_res4")

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("mcphersonsharyn_silver.cc_silver")

print(f"Rows written: {df.count()}")
# Removes negative values and -999 sentinel values (cc_raw > 0)
# Removes physically impossible values above 100 mg/m3
# Latitude filter applied — no longitude filter needed
# H3 resolution 4 for joining to SST Silver

In [0]:
%sql
SELECT
  'cc_silver' AS table_name,
  count(*) AS total_rows,
  count(DISTINCT year) AS distinct_years,
  min(year) AS earliest_year,
  max(year) AS latest_year,
  count(DISTINCT month) AS distinct_months,
  round(min(chlorophyll_concentration), 4) AS min_cc,
  round(max(chlorophyll_concentration), 4) AS max_cc,
  round(avg(chlorophyll_concentration), 4) AS avg_cc,
  count(CASE WHEN h3_res4 IS NULL THEN 1 END) AS null_h3
FROM mcphersonsharyn_silver.cc_silver;
-- Confirms CC Silver built correctly
-- min_cc should be above 0
-- max_cc should be below 100
-- avg_cc around 0.3 to 0.5 typical for tropical waters
-- null_h3 should be 0


In [0]:
%python
from pyspark.sql.functions import col, expr, sqrt, pow as spark_pow, round as spark_round

df = spark.table("mcphersonsharyn_bronze.bronze_cmems_raw") \
    .filter(col("uo").isNotNull()) \
    .filter((col("so") >= 0) & (col("so") <= 45)) \
    .filter((col("latitude") >= -23) & (col("latitude") <= 22)) \
    .filter((col("longitude") >= -73) & (col("longitude") <= 126)) \
    .withColumn("year", expr("YEAR(time)")) \
    .withColumn("month", expr("MONTH(time)")) \
    .withColumn("current_east", col("uo").cast("double")) \
    .withColumn("current_north", col("vo").cast("double")) \
    .withColumn("current_speed",
        spark_round(sqrt(spark_pow(col("uo"), 2) + spark_pow(col("vo"), 2)), 4)) \
    .withColumn("sea_level_anomaly", col("zos").cast("double")) \
    .withColumn("mixed_layer_depth", col("mlotst").cast("double")) \
    .withColumn("salinity", col("so").cast("double")) \
    .withColumn("ocean_temperature", col("thetao").cast("double")) \
    .withColumn("h3_res4", expr("h3_longlatash3(longitude, latitude, 4)")) \
    .select("latitude", "longitude", "time", "year", "month",
            "current_east", "current_north", "current_speed",
            "sea_level_anomaly", "mixed_layer_depth",
            "salinity", "ocean_temperature", "h3_res4")

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("mcphersonsharyn_silver.cmems_silver")

print(f"Rows written: {df.count()}")
# current_speed derived using Pythagoras theorem sqrt(uo2 + vo2)
# Salinity filter 0 to 45 PSU removes impossible values
# Bounding box applied — CMEMS

In [0]:
%sql
SELECT
  'cmems_silver' AS table_name,
  count(*) AS total_rows,
  count(DISTINCT year) AS distinct_years,
  min(year) AS earliest_year,
  max(year) AS latest_year,
  round(avg(current_speed), 4) AS avg_current_speed,
  round(avg(mixed_layer_depth), 2) AS avg_mld,
  round(avg(salinity), 2) AS avg_salinity,
  count(CASE WHEN h3_res4 IS NULL THEN 1 END) AS null_h3
FROM mcphersonsharyn_silver.cmems_silver;
-- Confirms CMEMS Silver built correctly
-- Expected: 322 million rows, 34 distinct years 1993 to 2026
-- avg_salinity around 34 to 36 PSU typical for tropical ocean
-- null_h3 should be 0

In [0]:
%sql
CREATE OR REPLACE TABLE mcphersonsharyn_silver.enso_silver
USING DELTA AS
SELECT YEAR AS year, 1 AS month, DJ AS mei_value FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE DJ IS NOT NULL AND DJ != -999
UNION ALL
SELECT YEAR, 2, JF FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE JF IS NOT NULL AND JF != -999
UNION ALL
SELECT YEAR, 3, FM FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE FM IS NOT NULL AND FM != -999
UNION ALL
SELECT YEAR, 4, MA FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE MA IS NOT NULL AND MA != -999
UNION ALL
SELECT YEAR, 5, AM FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE AM IS NOT NULL AND AM != -999
UNION ALL
SELECT YEAR, 6, MJ FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE MJ IS NOT NULL AND MJ != -999
UNION ALL
SELECT YEAR, 7, JJ FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE JJ IS NOT NULL AND JJ != -999
UNION ALL
SELECT YEAR, 8, JA FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE JA IS NOT NULL AND JA != -999
UNION ALL
SELECT YEAR, 9, AS_ FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE AS_ IS NOT NULL AND AS_ != -999
UNION ALL
SELECT YEAR, 10, SO FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE SO IS NOT NULL AND SO != -999
UNION ALL
SELECT YEAR, 11, ON_ FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE ON_ IS NOT NULL AND ON_ != -999
UNION ALL
SELECT YEAR, 12, ND FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE ND IS NOT NULL AND ND != -999;
-- Unpivots ENSO from wide format to long format
-- One row per year per month
-- Each bi-monthly season mapped to its primary calendar month
-- Expected: approximately 552 rows (46 years x 12 months)

In [0]:
%sql
SELECT *
FROM mcphersonsharyn_bronze.bronze_enso_raw
LIMIT 3;
-- Shows actual column names in ENSO Bronze table
-- Column names differ from expected DJ, JF, FM format

In [0]:
%sql
CREATE OR REPLACE TABLE mcphersonsharyn_silver.enso_silver
USING DELTA AS
SELECT year, 1 AS month, jan AS mei_value,
  CASE WHEN jan > 0.5 THEN 'El Nino' WHEN jan < -0.5 THEN 'La Nina' ELSE 'Neutral' END AS enso_phase
FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE jan IS NOT NULL
UNION ALL
SELECT year, 2, feb,
  CASE WHEN feb > 0.5 THEN 'El Nino' WHEN feb < -0.5 THEN 'La Nina' ELSE 'Neutral' END
FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE feb IS NOT NULL
UNION ALL
SELECT year, 3, mar,
  CASE WHEN mar > 0.5 THEN 'El Nino' WHEN mar < -0.5 THEN 'La Nina' ELSE 'Neutral' END
FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE mar IS NOT NULL
UNION ALL
SELECT year, 4, apr,
  CASE WHEN apr > 0.5 THEN 'El Nino' WHEN apr < -0.5 THEN 'La Nina' ELSE 'Neutral' END
FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE apr IS NOT NULL
UNION ALL
SELECT year, 5, may,
  CASE WHEN may > 0.5 THEN 'El Nino' WHEN may < -0.5 THEN 'La Nina' ELSE 'Neutral' END
FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE may IS NOT NULL
UNION ALL
SELECT year, 6, jun,
  CASE WHEN jun > 0.5 THEN 'El Nino' WHEN jun < -0.5 THEN 'La Nina' ELSE 'Neutral' END
FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE jun IS NOT NULL
UNION ALL
SELECT year, 7, jul,
  CASE WHEN jul > 0.5 THEN 'El Nino' WHEN jul < -0.5 THEN 'La Nina' ELSE 'Neutral' END
FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE jul IS NOT NULL
UNION ALL
SELECT year, 8, aug,
  CASE WHEN aug > 0.5 THEN 'El Nino' WHEN aug < -0.5 THEN 'La Nina' ELSE 'Neutral' END
FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE aug IS NOT NULL
UNION ALL
SELECT year, 9, sep,
  CASE WHEN sep > 0.5 THEN 'El Nino' WHEN sep < -0.5 THEN 'La Nina' ELSE 'Neutral' END
FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE sep IS NOT NULL
UNION ALL
SELECT year, 10, oct,
  CASE WHEN oct > 0.5 THEN 'El Nino' WHEN oct < -0.5 THEN 'La Nina' ELSE 'Neutral' END
FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE oct IS NOT NULL
UNION ALL
SELECT year, 11, nov,
  CASE WHEN nov > 0.5 THEN 'El Nino' WHEN nov < -0.5 THEN 'La Nina' ELSE 'Neutral' END
FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE nov IS NOT NULL
UNION ALL
SELECT year, 12, dec,
  CASE WHEN dec > 0.5 THEN 'El Nino' WHEN dec < -0.5 THEN 'La Nina' ELSE 'Neutral' END
FROM mcphersonsharyn_bronze.bronze_enso_raw WHERE dec IS NOT NULL;
-- Unpivots monthly columns to long format one row per year per month
-- ENSO phase classification applied directly in UNION ALL
-- El Nino MEI > 0.5, La Nina MEI < -0.5, Neutral between
-- Expected: approximately 552 rows (46 years x 12 months)

In [0]:
%python
from pyspark.sql.functions import lit, when, col

df = spark.table("mcphersonsharyn_bronze.bronze_enso_raw")

months = {
    'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4,
    'may': 5, 'jun': 6, 'jul': 7, 'aug': 8,
    'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
}

dfs = []
for month_name, month_num in months.items():
    df_month = df.filter(col(month_name).isNotNull()) \
        .select(
            col("year"),
            lit(month_num).alias("month"),
            col(month_name).cast("double").alias("mei_value")
        ) \
        .withColumn("enso_phase",
            when(col("mei_value") > 0.5, "El Nino")
            .when(col("mei_value") < -0.5, "La Nina")
            .otherwise("Neutral")
        )
    dfs.append(df_month)

from functools import reduce
from pyspark.sql import DataFrame
df_all = reduce(DataFrame.union, dfs)

df_all.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("mcphersonsharyn_silver.enso_silver")

print(f"Rows written: {df_all.count()}")
# Unpivots ENSO monthly columns to long format using Python
# One row per year per month with MEI value and phase label
# Expected: approximately 552 rows

In [0]:
%sql
SELECT
  'enso_silver' AS table_name,
  count(*) AS total_rows,
  count(DISTINCT year) AS distinct_years,
  min(year) AS earliest_year,
  max(year) AS latest_year,
  count(CASE WHEN enso_phase = 'El Nino' THEN 1 END) AS el_nino_months,
  count(CASE WHEN enso_phase = 'La Nina' THEN 1 END) AS la_nina_months,
  count(CASE WHEN enso_phase = 'Neutral' THEN 1 END) AS neutral_months
FROM mcphersonsharyn_silver.enso_silver;
-- Confirms 576 rows across 48 years x 12 months
-- Three phases should all be present
-- Distribution shows ENSO history 1979 to 2026

In [0]:
%sql
SELECT *
FROM mcphersonsharyn_bronze.bronze_microplastics_raw
LIMIT 3;
-- Shows exact column names in NCEI microplastics Bronze table
-- Need these before building the UNION ALL query

In [0]:
%sql
DESCRIBE TABLE mcphersonsharyn_bronze.bronze_microplastics_raw;
-- Shows all column names and data types
-- Need to find the concentration and date column names

In [0]:
%python
from pyspark.sql.functions import col, lit, expr, to_date, year, month

# NCEI Atlantic microplastics
ncei = spark.table("mcphersonsharyn_bronze.bronze_microplastics_raw") \
    .filter(col("latitude_degree").between(-23, 22)) \
    .filter(col("longitude_degree").between(-180, 180)) \
    .filter(col("microplastics_measurement") >= 0) \
    .filter(col("microplastics_measurement").isNotNull()) \
    .withColumn("sample_date_parsed", to_date(col("sample_date"))) \
    .withColumn("year", year(col("sample_date_parsed"))) \
    .withColumn("month", month(col("sample_date_parsed"))) \
    .withColumn("microplastic_concentration", col("microplastics_measurement").cast("double")) \
    .withColumn("data_source", lit("NCEI")) \
    .withColumn("h3_res4", expr("h3_longlatash3(longitude_degree, latitude_degree, 4)")) \
    .select("latitude_degree", "longitude_degree", "sample_date_parsed",
            "year", "month", "microplastic_concentration", "data_source", "h3_res4") \
    .withColumnRenamed("latitude_degree", "latitude") \
    .withColumnRenamed("longitude_degree", "longitude") \
    .withColumnRenamed("sample_date_parsed", "sample_date")

# AOMI Indo-Pacific microplastics
aomi = spark.table("mcphersonsharyn_bronze.bronze_microplastics_aomi_raw") \
    .filter(col("latitude_raw").cast("double").between(-23, 22)) \
    .filter(col("particle_density_raw").cast("double") >= 0) \
    .filter(col("particle_density_raw").isNotNull()) \
    .withColumn("latitude", col("latitude_raw").cast("double")) \
    .withColumn("longitude", col("longitude_raw").cast("double")) \
    .withColumn("sample_date", to_date(col("sample_date_raw"))) \
    .withColumn("year", year(to_date(col("sample_date_raw")))) \
    .withColumn("month", month(to_date(col("sample_date_raw")))) \
    .withColumn("microplastic_concentration", col("particle_density_raw").cast("double")) \
    .withColumn("data_source", lit("AOMI")) \
    .withColumn("h3_res4", expr("h3_longlatash3(longitude, latitude, 4)")) \
    .select("latitude", "longitude", "sample_date",
            "year", "month", "microplastic_concentration", "data_source", "h3_res4")

# Combine both sources
df_combined = ncei.union(aomi)

df_combined.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("mcphersonsharyn_silver.microplastics_silver")

print(f"Rows written: {df_combined.count()}")
print(f"NCEI rows: {ncei.count()}")
print(f"AOMI rows: {aomi.count()}")
# Combines NCEI Atlantic and AOMI Indo-Pacific microplastics
# UNION joins both datasets into single Silver table
# data_source column tracks which dataset each row came from
# H3 resolution 4 for joining to environmental conditions

In [0]:
%python
spark.sql("DROP TABLE IF EXISTS mcphersonsharyn_silver.microplastics_silver")
print("Table dropped")

In [0]:
%python
from pyspark.sql.functions import col, lit, expr, to_date, year, month, coalesce

spark.sql("DROP TABLE IF EXISTS mcphersonsharyn_silver.microplastics_silver")

# NCEI Atlantic microplastics
# Date format is M/d/yyyy h:mm:ss a e.g. 6/3/2014 12:00:00 AM
ncei = spark.table("mcphersonsharyn_bronze.bronze_microplastics_raw") \
    .filter(col("latitude_degree").between(-23, 22)) \
    .filter(col("microplastics_measurement") >= 0) \
    .filter(col("microplastics_measurement").isNotNull()) \
    .withColumn("sample_date",
        coalesce(
            to_date(col("sample_date"), "M/d/yyyy h:mm:ss a"),
            to_date(col("sample_date"), "yyyy-MM-dd"),
            to_date(col("sample_date"), "M/d/yyyy")
        )
    ) \
    .withColumn("year", year(col("sample_date"))) \
    .withColumn("month", month(col("sample_date"))) \
    .withColumn("microplastic_concentration",
        col("microplastics_measurement").cast("double")) \
    .withColumn("data_source", lit("NCEI")) \
    .withColumn("h3_res4",
        expr("h3_longlatash3(longitude_degree, latitude_degree, 4)")) \
    .select(
        col("latitude_degree").alias("latitude"),
        col("longitude_degree").alias("longitude"),
        "sample_date", "year", "month",
        "microplastic_concentration", "data_source", "h3_res4"
    )

# AOMI Indo-Pacific microplastics
aomi = spark.table("mcphersonsharyn_bronze.bronze_microplastics_aomi_raw") \
    .withColumn("latitude", col("latitude_raw").cast("double")) \
    .withColumn("longitude", col("longitude_raw").cast("double")) \
    .filter(col("latitude").between(-23, 22)) \
    .filter(col("particle_density_raw").cast("double") >= 0) \
    .filter(col("particle_density_raw").isNotNull()) \
    .withColumn("sample_date", to_date(col("sample_date_raw"))) \
    .withColumn("year", year(to_date(col("sample_date_raw")))) \
    .withColumn("month", month(to_date(col("sample_date_raw")))) \
    .withColumn("microplastic_concentration",
        col("particle_density_raw").cast("double")) \
    .withColumn("data_source", lit("AOMI")) \
    .withColumn("h3_res4",
        expr("h3_longlatash3(longitude, latitude, 4)")) \
    .select("latitude", "longitude", "sample_date",
            "year", "month", "microplastic_concentration",
            "data_source", "h3_res4")

df_combine

In [0]:
%python
from pyspark.sql.functions import col, lit, expr, to_date, year, month, coalesce

spark.sql("DROP TABLE IF EXISTS mcphersonsharyn_silver.microplastics_silver")

# NCEI Atlantic microplastics
ncei = spark.table("mcphersonsharyn_bronze.bronze_microplastics_raw") \
    .filter(col("latitude_degree").between(-23, 22)) \
    .filter(col("microplastics_measurement") >= 0) \
    .filter(col("microplastics_measurement").isNotNull()) \
    .withColumn("sample_date",
        coalesce(
            to_date(col("sample_date"), "M/d/yyyy h:mm:ss a"),
            to_date(col("sample_date"), "yyyy-MM-dd"),
            to_date(col("sample_date"), "M/d/yyyy")
        )) \
    .withColumn("year", year(col("sample_date"))) \
    .withColumn("month", month(col("sample_date"))) \
    .withColumn("microplastic_concentration",
        col("microplastics_measurement").cast("double")) \
    .withColumn("data_source", lit("NCEI")) \
    .withColumn("h3_res4",
        expr("h3_longlatash3(longitude_degree, latitude_degree, 4)")) \
    .select(
        col("latitude_degree").alias("latitude"),
        col("longitude_degree").alias("longitude"),
        "sample_date", "year", "month",
        "microplastic_concentration", "data_source", "h3_res4"
    )

# AOMI Indo-Pacific microplastics
aomi = spark.table("mcphersonsharyn_bronze.bronze_microplastics_aomi_raw") \
    .withColumn("latitude", col("latitude_raw").cast("double")) \
    .withColumn("longitude", col("longitude_raw").cast("double")) \
    .filter(col("latitude").between(-23, 22)) \
    .filter(col("particle_density_raw").cast("double") >= 0) \
    .filter(col("particle_density_raw").isNotNull()) \
    .withColumn("sample_date", to_date(col("sample_date_raw"))) \
    .withColumn("year", year(to_date(col("sample_date_raw")))) \
    .withColumn("month", month(to_date(col("sample_date_raw")))) \
    .withColumn("microplastic_concentration",
        col("particle_density_raw").cast("double")) \
    .withColumn("data_source", lit("AOMI")) \
    .withColumn("h3_res4",
        expr("h3_longlatash3(longitude, latitude, 4)")) \
    .select("latitude", "longitude", "sample_date",
            "year", "month", "microplastic_concentration",
            "data_source", "h3_res4")

df_combined = ncei.union(aomi)

df_combined.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("mcphersonsharyn_silver.microplastics_silver")

print(f"Total rows written: {df_combined.count()}")
print(f"NCEI rows: {ncei.count()}")
print(f"AOMI rows: {aomi.count()}")
# Fixed variable name df_combined consistently throughout
# coalesce handles multiple NCEI date formats

In [0]:
%python
from pyspark.sql.functions import avg, count, round as spark_round, col

df_agg = spark.table("mcphersonsharyn_silver.microplastics_silver") \
    .filter(col("microplastic_concentration").isNotNull()) \
    .groupBy("h3_res4", "month", "data_source") \
    .agg(
        spark_round(avg("microplastic_concentration"), 4).alias("microplastic_concentration"),
        count("*").alias("sample_count")
    )

df_agg.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("mcphersonsharyn_silver.microplastics_h3_monthly")

print(f"H3 monthly aggregation rows: {df_agg.count()}")
# Aggregates individual measurements to monthly H3 cell averages
# month join only (not year) because microplastics data is sparse
# sample_count shows how many measurements went into each average
# This table joins to environmental_conditions_silver on h3_res4 and month

In [0]:
%python
spark.sql("DROP TABLE IF EXISTS mcphersonsharyn_silver.microplastics_h3_monthly")
print("Table dropped")

In [0]:
%python
from pyspark.sql.functions import avg, count, round as spark_round, col

df_agg = spark.table("mcphersonsharyn_silver.microplastics_silver") \
    .filter(col("microplastic_concentration").isNotNull()) \
    .groupBy("h3_res4", "month", "data_source") \
    .agg(
        spark_round(avg("microplastic_concentration"), 4).alias("microplastic_concentration"),
        count("*").alias("sample_count")
    )

df_agg.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("mcphersonsharyn_silver.microplastics_h3_monthly")

print(f"H3 monthly aggregation rows: {df_agg.count()}")
# Aggregates individual measurements to monthly H3 cell averages
# Groups by H3 cell, month and data source
# sample_count tracks how many observations per cell

In [0]:
%sql
SELECT
  data_source,
  count(*) AS total_rows,
  round(avg(microplastic_concentration), 4) AS avg_concentration,
  round(max(microplastic_concentration), 4) AS max_concentration,
  min(year) AS earliest_year,
  max(year) AS latest_year
FROM mcphersonsharyn_silver.microplastics_silver
GROUP BY data_source;
-- Shows NCEI and AOMI rows separately
-- Confirms both sources present with valid concentration ranges
-- AOMI should show Indo-Pacific dates, NCEI Atlantic dates

In [0]:
%python
# Check AOMI concentration values
spark.table("mcphersonsharyn_bronze.bronze_microplastics_aomi_raw") \
    .select("particle_density_raw") \
    .show(10)
# Shows raw particle density values
# NaN suggests the values are stored as string 'NaN' not null

In [0]:
%python
# Check if NaN values exist in microplastics_silver for AOMI
spark.table("mcphersonsharyn_silver.microplastics_silver") \
    .filter("data_source = 'AOMI'") \
    .selectExpr(
        "count(*) as total",
        "count(microplastic_concentration) as non_null",
        "sum(case when isnan(microplastic_concentration) then 1 else 0 end) as nan_count",
        "avg(microplastic_concentration) as avg_conc",
        "min(microplastic_concentration) as min_conc",
        "max(microplastic_concentration) as max_conc"
    ).show()
# Checks for actual NaN values in the Silver table
# NaN in avg usually means NaN values exist in the data
# isnan() detects IEEE 754 NaN distinct from null

In [0]:
%python
from pyspark.sql.functions import col, lit, expr, to_date, year, month, coalesce, isnan

spark.sql("DROP TABLE IF EXISTS mcphersonsharyn_silver.microplastics_silver")
spark.sql("DROP TABLE IF EXISTS mcphersonsharyn_silver.microplastics_h3_monthly")

# NCEI Atlantic microplastics
ncei = spark.table("mcphersonsharyn_bronze.bronze_microplastics_raw") \
    .filter(col("latitude_degree").between(-23, 22)) \
    .filter(col("microplastics_measurement") >= 0) \
    .filter(col("microplastics_measurement").isNotNull()) \
    .filter(~isnan(col("microplastics_measurement"))) \
    .withColumn("sample_date",
        coalesce(
            to_date(col("sample_date"), "M/d/yyyy h:mm:ss a"),
            to_date(col("sample_date"), "yyyy-MM-dd"),
            to_date(col("sample_date"), "M/d/yyyy")
        )) \
    .withColumn("year", year(col("sample_date"))) \
    .withColumn("month", month(col("sample_date"))) \
    .withColumn("microplastic_concentration",
        col("microplastics_measurement").cast("double")) \
    .withColumn("data_source", lit("NCEI")) \
    .withColumn("h3_res4",
        expr("h3_longlatash3(longitude_degree, latitude_degree, 4)")) \
    .select(
        col("latitude_degree").alias("latitude"),
        col("longitude_degree").alias("longitude"),
        "sample_date", "year", "month",
        "microplastic_concentration", "data_source", "h3_res4"
    )

# AOMI Indo-Pacific microplastics
aomi = spark.table("mcphersonsharyn_bronze.bronze_microplastics_aomi_raw") \
    .withColumn("latitude", col("latitude_raw").cast("double")) \
    .withColumn("longitude", col("longitude_raw").cast("double")) \
    .withColumn("concentration", col("particle_density_raw").cast("double")) \
    .filter(col("latitude").between(-23, 22)) \
    .filter(col("concentration") >= 0) \
    .filter(col("concentration").isNotNull()) \
    .filter(~isnan(col("concentration"))) \
    .withColumn("sample_date", to_date(col("sample_date_raw"))) \
    .withColumn("year", year(to_date(col("sample_date_raw")))) \
    .withColumn("month", month(to_date(col("sample_date_raw")))) \
    .withColumn("data_source", lit("AOMI")) \
    .withColumn("h3_res4",
        expr("h3_longlatash3(longitude, latitude, 4)")) \
    .select("latitude", "longitude", "sample_date",
            "year", "month",
            col("concentration").alias("microplastic_concentration"),
            "data_source", "h3_res4")

df_combined = ncei.union(aomi)

df_combined.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("mcphersonsharyn_silver.microplastics_silver")

# Rebuild H3 monthly aggregation
from pyspark.sql.functions import avg, count, round as spark_round

df_agg = spark.table("mcphersonsharyn_silver.microplastics_silver") \
    .filter(col("microplastic_concentration").isNotNull()) \
    .filter(~isnan(col("microplastic_concentration"))) \
    .groupBy("h3_res4", "month", "data_source") \
    .agg(
        spark_round(avg("microplastic_concentration"), 4).alias("microplastic_concentration"),
        count("*").alias("sample_count")
    )

df_agg.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("mcphersonsharyn_silver.microplastics_h3_monthly")

print(f"Microplastics Silver rows: {df_combined.count()}")
print(f"H3 monthly rows: {df_agg.count()}")
# isnan() filter removes IEEE 754 NaN values distinct from null
# Both tables rebuilt cleanly without NaN contamination

In [0]:
%sql
SELECT
  data_source,
  count(*) AS total_rows,
  round(avg(microplastic_concentration), 4) AS avg_concentration,
  round(max(microplastic_concentration), 4) AS max_concentration,
  min(year) AS earliest_year,
  max(year) AS latest_year
FROM mcphersonsharyn_silver.microplastics_silver
GROUP BY data_source;
-- Confirms AOMI avg now shows real number not NaN
-- 5 NaN rows removed from AOMI
-- NCEI high values expected for coastal and beach samples

In [0]:
%python
from pyspark.sql.functions import col, expr, to_date, year, month, when

df = spark.table("mcphersonsharyn_bronze.bronze_coral_bleaching_raw") \
    .filter(col("time") != "UTC") \
    .withColumn("observation_date",
        to_date(col("time"), "yyyy-MM-dd'T'HH:mm:ss'Z'")) \
    .withColumn("year", year(col("observation_date"))) \
    .withColumn("month", month(col("observation_date"))) \
    .withColumn("latitude_d", col("latitude").cast("double")) \
    .withColumn("longitude_d", col("longitude").cast("double")) \
    .withColumn("degree_heating_week", col("crw_dhw").cast("double")) \
    .withColumn("bleaching_alert_area", col("crw_baa").cast("integer")) \
    .filter(col("latitude_d").between(-23, 22)) \
    .filter(col("longitude_d").between(-180, 180)) \
    .filter(col("degree_heating_week") >= 0) \
    .filter(col("degree_heating_week").isNotNull()) \
    .withColumn("bleaching_alert_category",
        when(col("bleaching_alert_area") == 0, "No Stress")
        .when(col("bleaching_alert_area") == 1, "Watch")
        .when(col("bleaching_alert_area") == 2, "Warning")
        .when(col("bleaching_alert_area") == 3, "Alert Level 1")
        .when(col("bleaching_alert_area") == 4, "Alert Level 2")
        .when(col("bleaching_alert_area") == 5, "Alert Level 2+")
        .otherwise("Unknown")
    ) \
    .withColumn("h3_res4",
        expr("h3_longlatash3(longitude_d, latitude_d, 4)")) \
    .select(
        col("latitude_d").alias("latitude"),
        col("longitude_d").alias("longitude"),
        "observation_date", "year", "month",
        "degree_heating_week", "bleaching_alert_area",
        "bleaching_alert_category", "h3_res4"
    )

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("mcphersonsharyn_silver.coral_bleaching_silver")

print(f"Rows written: {df.count()}")
# Parses ISO timestamp format 1985-04-01T12:00:00Z to DATE
# Adds BAA category label for plain English interpretation
# DHW filter removes negative values and nulls
# H3 resolution 4 for joining to environmental conditions

In [0]:
%python
from pyspark.sql.functions import col, expr, to_date, year, month, when
from pyspark.sql.functions import try_cast

spark.sql("DROP TABLE IF EXISTS mcphersonsharyn_silver.coral_bleaching_silver")

df = spark.table("mcphersonsharyn_bronze.bronze_coral_bleaching_raw") \
    .filter(col("time") != "UTC") \
    .withColumn("observation_date",
        to_date(col("time"), "yyyy-MM-dd'T'HH:mm:ss'Z'")) \
    .withColumn("year", year(col("observation_date"))) \
    .withColumn("month", month(col("observation_date"))) \
    .withColumn("latitude_d", col("latitude").cast("double")) \
    .withColumn("longitude_d", col("longitude").cast("double")) \
    .withColumn("degree_heating_week", col("crw_dhw").cast("double")) \
    .withColumn("bleaching_alert_area",
        expr("try_cast(crw_baa AS INT)")) \
    .filter(col("latitude_d").between(-23, 22)) \
    .filter(col("longitude_d").between(-180, 180)) \
    .filter(col("degree_heating_week") >= 0) \
    .filter(col("degree_heating_week").isNotNull()) \
    .withColumn("bleaching_alert_category",
        when(col("bleaching_alert_area") == 0, "No Stress")
        .when(col("bleaching_alert_area") == 1, "Watch")
        .when(col("bleaching_alert_area") == 2, "Warning")
        .when(col("bleaching_alert_area") == 3, "Alert Level 1")
        .when(col("bleaching_alert_area") == 4, "Alert Level 2")
        .when(col("bleaching_alert_area") == 5, "Alert Level 2+")
        .otherwise("Unknown")
    ) \
    .withColumn("h3_res4",
        expr("h3_longlatash3(longitude_d, latitude_d, 4)")) \
    .select(
        col("latitude_d").alias("latitude"),
        col("longitude_d").alias("longitude"),
        "observation_date", "year", "month",
        "degree_heating_week", "bleaching_alert_area",
        "bleaching_alert_category", "h3_res4"
    )

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("mcphersonsharyn_silver.coral_bleaching_silver")

print(f"Rows written: {df.count()}")
# try_cast handles NaN values gracefully converting them to NULL
# NaN in crw_baa becomes NULL bleaching_alert_area in Silver
# NULL BAA rows categorised as Unknown in bleaching_alert_category

In [0]:
%python
from pyspark.sql.functions import col, expr, to_date, year, month, when

spark.sql("DROP TABLE IF EXISTS mcphersonsharyn_silver.coral_bleaching_silver")

df = spark.table("mcphersonsharyn_bronze.bronze_coral_bleaching_raw") \
    .filter(col("time") != "UTC") \
    .withColumn("observation_date",
        to_date(col("time"), "yyyy-MM-dd'T'HH:mm:ss'Z'")) \
    .withColumn("year", year(col("observation_date"))) \
    .withColumn("month", month(col("observation_date"))) \
    .withColumn("latitude_d", col("latitude").cast("double")) \
    .withColumn("longitude_d", col("longitude").cast("double")) \
    .withColumn("degree_heating_week",
        expr("try_cast(crw_dhw AS DOUBLE)")) \
    .withColumn("bleaching_alert_area",
        expr("try_cast(crw_baa AS INT)")) \
    .filter(col("latitude_d").between(-23, 22)) \
    .filter(col("longitude_d").between(-180, 180)) \
    .filter(col("degree_heating_week") >= 0) \
    .filter(col("degree_heating_week").isNotNull()) \
    .withColumn("bleaching_alert_category",
        when(col("bleaching_alert_area") == 0, "No Stress")
        .when(col("bleaching_alert_area") == 1, "Watch")
        .when(col("bleaching_alert_area") == 2, "Warning")
        .when(col("bleaching_alert_area") == 3, "Alert Level 1")
        .when(col("bleaching_alert_area") == 4, "Alert Level 2")
        .when(col("bleaching_alert_area") == 5, "Alert Level 2+")
        .otherwise("Unknown")
    ) \
    .withColumn("h3_res4",
        expr("h3_longlatash3(longitude_d, latitude_d, 4)")) \
    .select(
        col("latitude_d").alias("latitude"),
        col("longitude_d").alias("longitude"),
        "observation_date", "year", "month",
        "degree_heating_week", "bleaching_alert_area",
        "bleaching_alert_category", "h3_res4"
    )

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("mcphersonsharyn_silver.coral_bleaching_silver")

print(f"Rows written: {df.count()}")
# expr("try_cast(...)") calls Databricks SQL try_cast from Python
# Handles NaN values in crw_baa converting to NULL safely
# All other transformations same as before

In [0]:
%sql
-- Drop and recreate with correct schema first
DROP TABLE IF EXISTS mcphersonsharyn_silver.coral_bleaching_silver;

CREATE TABLE mcphersonsharyn_silver.coral_bleaching_silver
(
  latitude DOUBLE,
  longitude DOUBLE,
  observation_date DATE,
  year INT,
  month INT,
  degree_heating_week DOUBLE,
  bleaching_alert_area INT,
  bleaching_alert_category STRING,
  h3_res4 STRING
)
USING DELTA;
-- Creates empty table with correct schema first
-- Then we INSERT INTO from Bronze

In [0]:
%sql
INSERT INTO mcphersonsharyn_silver.coral_bleaching_silver
SELECT
  TRY_CAST(latitude AS DOUBLE) AS latitude,
  TRY_CAST(longitude AS DOUBLE) AS longitude,
  TO_DATE(time, "yyyy-MM-dd'T'HH:mm:ss'Z'") AS observation_date,
  YEAR(TO_DATE(time, "yyyy-MM-dd'T'HH:mm:ss'Z'")) AS year,
  MONTH(TO_DATE(time, "yyyy-MM-dd'T'HH:mm:ss'Z'")) AS month,
  TRY_CAST(crw_dhw AS DOUBLE) AS degree_heating_week,
  TRY_CAST(crw_baa AS INT) AS bleaching_alert_area,
  CASE TRY_CAST(crw_baa AS INT)
    WHEN 0 THEN 'No Stress'
    WHEN 1 THEN 'Watch'
    WHEN 2 THEN 'Warning'
    WHEN 3 THEN 'Alert Level 1'
    WHEN 4 THEN 'Alert Level 2'
    WHEN 5 THEN 'Alert Level 2+'
    ELSE 'Unknown'
  END AS bleaching_alert_category,
  h3_longlatash3(
    TRY_CAST(longitude AS DOUBLE),
    TRY_CAST(latitude AS DOUBLE),
    4
  ) AS h3_res4
FROM mcphersonsharyn_bronze.bronze_coral_bleaching_raw
WHERE time != 'UTC'
AND TRY_CAST(latitude AS DOUBLE) BETWEEN -23 AND 22
AND TRY_CAST(crw_dhw AS DOUBLE) >= 0
AND TRY_CAST(crw_dhw AS DOUBLE) IS NOT NULL;
-- INSERT INTO works reliably where CREATE TABLE AS SELECT fails
-- TRY_CAST handles NaN values converting to NULL safely
-- This will take several minutes given 18 million rows

In [0]:
%sql
SELECT
  'coral_bleaching_silver' AS table_name,
  count(*) AS total_rows,
  count(DISTINCT year) AS distinct_years,
  min(year) AS earliest_year,
  max(year) AS latest_year,
  round(avg(degree_heating_week), 2) AS avg_dhw,
  round(max(degree_heating_week), 2) AS max_dhw,
  count(CASE WHEN bleaching_alert_area >= 2 THEN 1 END) AS bleaching_events,
  count(CASE WHEN h3_res4 IS NULL THEN 1 END) AS null_h3
FROM mcphersonsharyn_silver.coral_bleaching_silver;
-- Confirms coral bleaching Silver built correctly
-- Expected: annual snapshots 1985 to 2026
-- avg_dhw near 0 expected — most ocean not bleaching most of the time
-- bleaching_events shows observations above Warning threshold
-- null_h3 should be 0

In [0]:
%sql
SELECT
  count(*) AS total,
  count(degree_heating_week) AS non_null,
  count(CASE WHEN isnan(degree_heating_week) THEN 1 END) AS nan_count,
  count(CASE WHEN degree_heating_week IS NULL THEN 1 END) AS null_count
FROM mcphersonsharyn_silver.coral_bleaching_silver;
-- Checks for NaN values in degree_heating_week
-- NaN is distinct from NULL in floating point arithmetic
-- Even one NaN causes avg and max to return NaN

In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_silver.coral_bleaching_silver;

CREATE TABLE mcphersonsharyn_silver.coral_bleaching_silver
(
  latitude DOUBLE,
  longitude DOUBLE,
  observation_date DATE,
  year INT,
  month INT,
  degree_heating_week DOUBLE,
  bleaching_alert_area INT,
  bleaching_alert_category STRING,
  h3_res4 STRING
)
USING DELTA;
-- Recreating table to remove NaN DHW values
-- NaN occurs where bleaching cannot be calculated
-- typically open ocean areas far from coral reefs

In [0]:
%sql
INSERT INTO mcphersonsharyn_silver.coral_bleaching_silver
SELECT
  TRY_CAST(latitude AS DOUBLE) AS latitude,
  TRY_CAST(longitude AS DOUBLE) AS longitude,
  TO_DATE(time, "yyyy-MM-dd'T'HH:mm:ss'Z'") AS observation_date,
  YEAR(TO_DATE(time, "yyyy-MM-dd'T'HH:mm:ss'Z'")) AS year,
  MONTH(TO_DATE(time, "yyyy-MM-dd'T'HH:mm:ss'Z'")) AS month,
  TRY_CAST(crw_dhw AS DOUBLE) AS degree_heating_week,
  TRY_CAST(crw_baa AS INT) AS bleaching_alert_area,
  CASE TRY_CAST(crw_baa AS INT)
    WHEN 0 THEN 'No Stress'
    WHEN 1 THEN 'Watch'
    WHEN 2 THEN 'Warning'
    WHEN 3 THEN 'Alert Level 1'
    WHEN 4 THEN 'Alert Level 2'
    WHEN 5 THEN 'Alert Level 2+'
    ELSE 'Unknown'
  END AS bleaching_alert_category,
  h3_longlatash3(
    TRY_CAST(longitude AS DOUBLE),
    TRY_CAST(latitude AS DOUBLE),
    4
  ) AS h3_res4
FROM mcphersonsharyn_bronze.bronze_coral_bleaching_raw
WHERE time != 'UTC'
AND TRY_CAST(latitude AS DOUBLE) BETWEEN -23 AND 22
AND TRY_CAST(crw_dhw AS DOUBLE) >= 0
AND TRY_CAST(crw_dhw AS DOUBLE) IS NOT NULL
AND crw_dhw != 'NaN';
-- Added crw_dhw != 'NaN' to exclude NaN string values at source
-- NaN in Bronze means bleaching not calculable for that location
-- These are typically open ocean cells far from coral reefs
-- Removing them keeps only valid bleaching stress measurements

In [0]:
%sql
SELECT
  'coral_bleaching_silver' AS table_name,
  count(*) AS total_rows,
  count(DISTINCT year) AS distinct_years,
  min(year) AS earliest_year,
  max(year) AS latest_year,
  round(avg(degree_heating_week), 2) AS avg_dhw,
  round(max(degree_heating_week), 2) AS max_dhw,
  count(CASE WHEN bleaching_alert_area >= 2 THEN 1 END) AS bleaching_events,
  count(CASE WHEN h3_res4 IS NULL THEN 1 END) AS null_h3
FROM mcphersonsharyn_silver.coral_bleaching_silver;
-- Confirms NaN values removed
-- avg_dhw and max_dhw should now show real numbers
-- bleaching_events shows observations above Warning threshold


In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_silver.environmental_conditions_silver;

CREATE TABLE mcphersonsharyn_silver.environmental_conditions_silver
(
  h3_res4 STRING,
  year INT,
  month INT,
  latitude DOUBLE,
  longitude DOUBLE,
  sea_surface_temperature DOUBLE,
  chlorophyll_concentration DOUBLE,
  mei_value DOUBLE,
  enso_phase STRING,
  microplastic_concentration DOUBLE,
  microplastics_source STRING,
  current_speed DOUBLE,
  sea_level_anomaly DOUBLE,
  mixed_layer_depth DOUBLE,
  salinity DOUBLE,
  ocean_temperature DOUBLE,
  degree_heating_week DOUBLE,
  bleaching_alert_area INT,
  bleaching_alert_category STRING,
  habitat_stress_index INT
)
USING DELTA;
-- Master environmental join table
-- All six Silver environmental sources combined
-- Keyed by H3 cell, year and month
-- habitat_stress_index 0-4 calculated from four stressors

In [0]:
%sql
INSERT INTO mcphersonsharyn_silver.environmental_conditions_silver
SELECT
  sst.h3_res4,
  sst.year,
  sst.month,
  sst.latitude,
  sst.longitude,
  sst.sea_surface_temperature,
  cc.chlorophyll_concentration,
  enso.mei_value,
  enso.enso_phase,
  mp.microplastic_concentration,
  mp.data_source AS microplastics_source,
  cmems.current_speed,
  cmems.sea_level_anomaly,
  cmems.mixed_layer_depth,
  cmems.salinity,
  cmems.ocean_temperature,
  cb.degree_heating_week,
  cb.bleaching_alert_area,
  cb.bleaching_alert_category,
  CASE WHEN sst.sea_surface_temperature > 30 THEN 1 ELSE 0 END
  + CASE WHEN cc.chlorophyll_concentration < 0.1 THEN 1 ELSE 0 END
  + CASE WHEN mp.microplastic_concentration > 1.0 THEN 1 ELSE 0 END
  + CASE WHEN cb.bleaching_alert_area >= 2 THEN 1 ELSE 0 END
  AS habitat_stress_index
FROM mcphersonsharyn_silver.sst_silver sst
LEFT JOIN mcphersonsharyn_silver.cc_silver cc
  ON sst.h3_res4 = cc.h3_res4
  AND sst.year = cc.year
  AND sst.month = cc.month
LEFT JOIN mcphersonsharyn_silver.enso_silver enso
  ON sst.year = enso.year
  AND sst.month = enso.month
LEFT JOIN mcphersonsharyn_silver.microplastics_h3_monthly mp
  ON sst.h3_res4 = mp.h3_res4
  AND sst.month = mp.month
LEFT JOIN mcphersonsharyn_silver.cmems_silver cmems
  ON sst.h3_res4 = cmems.h3_res4
  AND sst.year = cmems.year
  AND sst.month = cmems.month
LEFT JOIN mcphersonsharyn_silver.coral_bleaching_silver cb
  ON sst.h3_res4 = cb.h3_res4
  AND sst.year = cb.year
  AND sst.month = cb.month;
-- SST is the primary table — all rows preserved via LEFT JOIN
-- ENSO joins on year and month only — global index no location
-- Microplastics joins on H3 and month only — sparse data
-- CMEMS and coral bleaching join on H3, year and month
-- Habitat stress index 0-4 calculated from four stressors
-- This will take several minutes given the volume of data

In [0]:
%sql
SELECT
  count(*) AS total_rows,
  count(DISTINCT year) AS distinct_years,
  min(year) AS earliest_year,
  max(year) AS latest_year,
  count(CASE WHEN sea_surface_temperature IS NULL THEN 1 END) AS null_sst,
  count(CASE WHEN chlorophyll_concentration IS NOT NULL THEN 1 END) AS rows_with_cc,
  count(CASE WHEN enso_phase IS NOT NULL THEN 1 END) AS rows_with_enso,
  count(CASE WHEN microplastic_concentration IS NOT NULL THEN 1 END) AS rows_with_mp,
  count(CASE WHEN current_speed IS NOT NULL THEN 1 END) AS rows_with_cmems,
  count(CASE WHEN degree_heating_week IS NOT NULL THEN 1 END) AS rows_with_bleaching,
  count(DISTINCT habitat_stress_index) AS stress_index_values
FROM mcphersonsharyn_silver.environmental_conditions_silver;
-- null_sst should be 0 — SST is the primary table
-- rows_with_enso should equal total_rows
-- stress_index_values should be 5 (0, 1, 2, 3, 4)

In [0]:
%sql
SELECT
  habitat_stress_index,
  count(*) AS rows
FROM mcphersonsharyn_silver.environmental_conditions_silver
GROUP BY habitat_stress_index
ORDER BY habitat_stress_index;
-- Shows distribution of habitat stress index values
-- Identifies which score is missing


In [0]:
%sql
-- NOTE: HSI 4 (all four stressors) not present in current dataset
-- Most likely reason: microplastics data is very sparse (1,871 matched rows)
-- The pollution stressor rarely overlaps with thermal + food + bleaching stress
-- simultaneously in the same H3 cell in the same month
-- This finding itself is conservation-relevant:
-- even the most stressed reef areas have not yet reached maximum composite stress
-- HSI 3 locations (1,473 rows) are the highest priority conservation concern

SELECT
  h3_res4,
  year,
  month,
  sea_surface_temperature,
  chlorophyll_concentration,
  degree_heating_week,
  bleaching_alert_category,
  habitat_stress_index
FROM mcphersonsharyn_silver.environmental_conditions_silver
WHERE habitat_stress_index = 3
ORDER BY year DESC
LIMIT 10;
-- Shows the most stressed locations in the dataset
-- These are the conservation priority H3 cells
-- Where are they? What year? What conditions?

In [0]:
%sql
-- Check how many duplicate rows exist
SELECT
  h3_res4, year, month,
  count(*) AS row_count
FROM mcphersonsharyn_silver.environmental_conditions_silver
GROUP BY h3_res4, year, month
HAVING count(*) > 1
ORDER BY row_count DESC
LIMIT 10;
-- Shows H3 cells with duplicate year/month combinations
-- If row_count > 1 we have duplicates from the join
-- Need to understand why before fixing

In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_silver.environmental_conditions_silver;

CREATE TABLE mcphersonsharyn_silver.environmental_conditions_silver
(
  h3_res4 STRING,
  year INT,
  month INT,
  latitude DOUBLE,
  longitude DOUBLE,
  sea_surface_temperature DOUBLE,
  chlorophyll_concentration DOUBLE,
  mei_value DOUBLE,
  enso_phase STRING,
  microplastic_concentration DOUBLE,
  microplastics_source STRING,
  current_speed DOUBLE,
  sea_level_anomaly DOUBLE,
  mixed_layer_depth DOUBLE,
  salinity DOUBLE,
  ocean_temperature DOUBLE,
  degree_heating_week DOUBLE,
  bleaching_alert_area INT,
  bleaching_alert_category STRING,
  habitat_stress_index INT
)
USING DELTA;
-- Recreating table with deduplication
-- CMEMS at 0.083 degree resolution produces multiple rows per H3 cell
-- ROW_NUMBER() keeps one row per H3 cell per year per month

In [0]:
%sql
INSERT INTO mcphersonsharyn_silver.environmental_conditions_silver
WITH ranked AS (
  SELECT
    sst.h3_res4,
    sst.year,
    sst.month,
    sst.latitude,
    sst.longitude,
    sst.sea_surface_temperature,
    cc.chlorophyll_concentration,
    enso.mei_value,
    enso.enso_phase,
    mp.microplastic_concentration,
    mp.data_source AS microplastics_source,
    cmems.current_speed,
    cmems.sea_level_anomaly,
    cmems.mixed_layer_depth,
    cmems.salinity,
    cmems.ocean_temperature,
    cb.degree_heating_week,
    cb.bleaching_alert_area,
    cb.bleaching_alert_category,
    CASE WHEN sst.sea_surface_temperature > 30 THEN 1 ELSE 0 END
    + CASE WHEN cc.chlorophyll_concentration < 0.1 THEN 1 ELSE 0 END
    + CASE WHEN mp.microplastic_concentration > 1.0 THEN 1 ELSE 0 END
    + CASE WHEN cb.bleaching_alert_area >= 2 THEN 1 ELSE 0 END
    AS habitat_stress_index,
    ROW_NUMBER() OVER (
      PARTITION BY sst.h3_res4, sst.year, sst.month
      ORDER BY sst.latitude, sst.longitude
    ) AS rn
  FROM mcphersonsharyn_silver.sst_silver sst
  LEFT JOIN mcphersonsharyn_silver.cc_silver cc
    ON sst.h3_res4 = cc.h3_res4
    AND sst.year = cc.year
    AND sst.month = cc.month
  LEFT JOIN mcphersonsharyn_silver.enso_silver enso
    ON sst.year = enso.year
    AND sst.month = enso.month
  LEFT JOIN mcphersonsharyn_silver.microplastics_h3_monthly mp
    ON sst.h3_res4 = mp.h3_res4
    AND sst.month = mp.month
  LEFT JOIN mcphersonsharyn_silver.cmems_silver cmems
    ON sst.h3_res4 = cmems.h3_res4
    AND sst.year = cmems.year
    AND sst.month = cmems.month
  LEFT JOIN mcphersonsharyn_silver.coral_bleaching_silver cb
    ON sst.h3_res4 = cb.h3_res4
    AND sst.year = cb.year
    AND sst.month = cb.month
)
SELECT
  h3_res4, year, month, latitude, longitude,
  sea_surface_temperature, chlorophyll_concentration,
  mei_value, enso_phase,
  microplastic_concentration, microplastics_source,
  current_speed, sea_level_anomaly, mixed_layer_depth,
  salinity, ocean_temperature,
  degree_heating_week, bleaching_alert_area, bleaching_alert_category,
  habitat_stress_index
FROM ranked
WHERE rn = 1;
-- ROW_NUMBER() deduplicates by keeping one row per H3 cell per year per month
-- PARTITION BY h3_res4, year, month ensures one row per time period per cell
-- ORDER BY latitude, longitude gives consistent row selection
-- WHERE rn = 1 keeps only the first row from each partition
-- This will take several minutes given the volume of data

In [0]:
%sql
SELECT
  count(*) AS total_rows,
  count(DISTINCT year) AS distinct_years,
  min(year) AS earliest_year,
  max(year) AS latest_year,
  count(CASE WHEN sea_surface_temperature IS NULL THEN 1 END) AS null_sst,
  count(CASE WHEN chlorophyll_concentration IS NOT NULL THEN 1 END) AS rows_with_cc,
  count(CASE WHEN enso_phase IS NOT NULL THEN 1 END) AS rows_with_enso,
  count(CASE WHEN current_speed IS NOT NULL THEN 1 END) AS rows_with_cmems,
  count(CASE WHEN degree_heating_week IS NOT NULL THEN 1 END) AS rows_with_bleaching,
  count(DISTINCT habitat_stress_index) AS stress_index_values
FROM mcphersonsharyn_silver.environmental_conditions_silver;
-- Confirms deduplication worked
-- total_rows should match SST Silver at 2,795,120
-- null_sst should be 0
-- rows_with_enso should equal total_rows

In [0]:
%sql
SELECT
  enso_phase,
  count(DISTINCT year) AS years_in_phase,
  round(avg(sea_surface_temperature), 2) AS avg_sst,
  round(avg(chlorophyll_concentration), 4) AS avg_cc,
  round(avg(degree_heating_week), 2) AS avg_dhw,
  round(avg(habitat_stress_index), 2) AS avg_stress
FROM mcphersonsharyn_silver.environmental_conditions_silver
WHERE enso_phase IS NOT NULL
AND sea_surface_temperature IS NOT NULL
GROUP BY enso_phase
ORDER BY avg_sst DESC;
-- THE KEY ATLAS FINDING QUERY
-- If La Nina avg_sst > El Nino avg_sst the finding is confirmed
-- across the full 30-year baseline with all new data sources
-- Previous finding: La Nina 29.47C vs El Nino 28.80C

In [0]:
%sql
-- Filter to Raja Ampat longitude range only
SELECT
  enso_phase,
  count(DISTINCT year) AS years_in_phase,
  round(avg(sea_surface_temperature), 2) AS avg_sst,
  round(avg(chlorophyll_concentration), 4) AS avg_cc
FROM mcphersonsharyn_silver.environmental_conditions_silver
WHERE enso_phase IS NOT NULL
AND sea_surface_temperature IS NOT NULL
AND longitude BETWEEN 129 AND 132
AND latitude BETWEEN -2 AND 1
GROUP BY enso_phase
ORDER BY avg_sst DESC;
-- Filters to Raja Ampat geographic area only
-- Longitude 129-132, Latitude -2 to 1
-- Tests whether the La Nina SST inversion holds in this specific region
-- Previous finding: La Nina 29.47C > El Nino 28.80C in Raja Ampat

In [0]:
%sql
-- Check H3 overlap between SST and CMEMS Silver
SELECT
  count(DISTINCT s.h3_res4) AS sst_h3_cells,
  count(DISTINCT c.h3_res4) AS cmems_h3_cells,
  count(DISTINCT s.h3_res4) - count(DISTINCT c.h3_res4) AS difference
FROM mcphersonsharyn_silver.sst_silver s
FULL OUTER JOIN mcphersonsharyn_silver.cmems_silver c
  ON s.h3_res4 = c.h3_res4;
-- Counts distinct H3 cells in each Silver table
-- If CMEMS has very few matching H3 cells the join will fail
-- The 0.083 degree CMEMS grid may not align with 0.25 degree SST H3 cells

In [0]:
%sql
-- Compare actual H3 values from SST and CMEMS
SELECT 'SST' AS source, h3_res4, latitude, longitude
FROM mcphersonsharyn_silver.sst_silver
WHERE year = 2009 AND month = 6
LIMIT 5;

In [0]:
%sql
SELECT 'CMEMS' AS source, h3_res4, latitude, longitude
FROM mcphersonsharyn_silver.cmems_silver
WHERE year = 2009 AND month = 6
LIMIT 5;
-- Compares actual H3 index values from both tables
-- If coordinates are similar but H3 values differ
-- the issue is coordinate precision affecting H3 calculation

In [0]:
%sql
SELECT
  min(longitude) AS min_lon,
  max(longitude) AS max_lon,
  min(latitude) AS min_lat,
  max(latitude) AS max_lat,
  count(DISTINCT longitude) AS distinct_lons,
  count(DISTINCT latitude) AS distinct_lats
FROM mcphersonsharyn_silver.sst_silver
WHERE year = 2009 AND month = 6;
-- Shows exact coordinate ranges in SST Silver
-- SST uses integer degree coordinates
-- CMEMS uses decimal degree coordinates
-- This explains why H3 cells do not overlap

In [0]:
%sql
-- Test H3 resolution 3 overlap
SELECT
  count(DISTINCT h3_longlatash3(longitude, latitude, 3)) AS sst_h3_res3_cells
FROM mcphersonsharyn_silver.sst_silver
WHERE year = 2009 AND month = 6;

In [0]:
%sql
SELECT
  count(DISTINCT h3_longlatash3(longitude, latitude, 3)) AS cmems_h3_res3_cells
FROM mcphersonsharyn_silver.cmems_silver
WHERE year = 2009 AND month = 6;
-- Tests whether resolution 3 gives better overlap between SST and CMEMS
-- Larger cells = more likely to match between coarse and fine grids


In [0]:
%sql
SELECT
  count(*) AS joined_rows
FROM mcphersonsharyn_silver.sst_silver sst
JOIN mcphersonsharyn_silver.cmems_silver cmems
  ON h3_longlatash3(sst.longitude, sst.latitude, 3) =
     h3_longlatash3(cmems.longitude, cmems.latitude, 3)
  AND sst.year = cmems.year
  AND sst.month = cmems.month
WHERE sst.year = 2009
AND sst.month = 6;
-- Tests join using resolution 3 calculated on the fly
-- If this returns many more rows than the current 20,400
-- we should rebuild environmental_conditions_silver using res3 for CMEMS join

In [0]:
%sql
-- Add h3_res3 column to SST Silver
ALTER TABLE mcphersonsharyn_silver.sst_silver
ADD COLUMN h3_res3 STRING;
-- Adds resolution 3 H3 column to SST Silver
-- Will be populated in the next step

In [0]:
%sql
-- Populate h3_res3 in SST Silver
UPDATE mcphersonsharyn_silver.sst_silver
SET h3_res3 = h3_longlatash3(longitude, latitude, 3);
-- Calculates H3 resolution 3 index for every SST row


In [0]:
%sql
ALTER TABLE mcphersonsharyn_silver.cmems_silver
ADD COLUMN h3_res3 STRING;
-- Adds resolution 3 H3 column to CMEMS Silver

In [0]:
%sql
UPDATE mcphersonsharyn_silver.cmems_silver
SET h3_res3 = h3_longlatash3(longitude, latitude, 3);
-- Calculates H3 resolution 3 index for every CMEMS row



In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_silver.environmental_conditions_silver;

CREATE TABLE mcphersonsharyn_silver.environmental_conditions_silver
(
  h3_res4 STRING,
  year INT,
  month INT,
  latitude DOUBLE,
  longitude DOUBLE,
  sea_surface_temperature DOUBLE,
  chlorophyll_concentration DOUBLE,
  mei_value DOUBLE,
  enso_phase STRING,
  microplastic_concentration DOUBLE,
  microplastics_source STRING,
  current_speed DOUBLE,
  sea_level_anomaly DOUBLE,
  mixed_layer_depth DOUBLE,
  salinity DOUBLE,
  ocean_temperature DOUBLE,
  degree_heating_week DOUBLE,
  bleaching_alert_area INT,
  bleaching_alert_category STRING,
  habitat_stress_index INT
)
USING DELTA;
-- Recreating with CMEMS joining on h3_res3 instead of h3_res4
-- Resolution 3 cells are large enough for both SST and CMEMS grids to overlap

In [0]:
%sql
INSERT INTO mcphersonsharyn_silver.environmental_conditions_silver
WITH ranked AS (
  SELECT
    sst.h3_res4,
    sst.year,
    sst.month,
    sst.latitude,
    sst.longitude,
    sst.sea_surface_temperature,
    cc.chlorophyll_concentration,
    enso.mei_value,
    enso.enso_phase,
    mp.microplastic_concentration,
    mp.data_source AS microplastics_source,
    cmems.current_speed,
    cmems.sea_level_anomaly,
    cmems.mixed_layer_depth,
    cmems.salinity,
    cmems.ocean_temperature,
    cb.degree_heating_week,
    cb.bleaching_alert_area,
    cb.bleaching_alert_category,
    CASE WHEN sst.sea_surface_temperature > 30 THEN 1 ELSE 0 END
    + CASE WHEN cc.chlorophyll_concentration < 0.1 THEN 1 ELSE 0 END
    + CASE WHEN mp.microplastic_concentration > 1.0 THEN 1 ELSE 0 END
    + CASE WHEN cb.bleaching_alert_area >= 2 THEN 1 ELSE 0 END
    AS habitat_stress_index,
    ROW_NUMBER() OVER (
      PARTITION BY sst.h3_res4, sst.year, sst.month
      ORDER BY sst.latitude, sst.longitude
    ) AS rn
  FROM mcphersonsharyn_silver.sst_silver sst
  LEFT JOIN mcphersonsharyn_silver.cc_silver cc
    ON sst.h3_res4 = cc.h3_res4
    AND sst.year = cc.year
    AND sst.month = cc.month
  LEFT JOIN mcphersonsharyn_silver.enso_silver enso
    ON sst.year = enso.year
    AND sst.month = enso.month
  LEFT JOIN mcphersonsharyn_silver.microplastics_h3_monthly mp
    ON sst.h3_res4 = mp.h3_res4
    AND sst.month = mp.month
  LEFT JOIN mcphersonsharyn_silver.cmems_silver cmems
    ON sst.h3_res3 = cmems.h3_res3
    AND sst.year = cmems.year
    AND sst.month = cmems.month
  LEFT JOIN mcphersonsharyn_silver.coral_bleaching_silver cb
    ON sst.h3_res4 = cb.h3_res4
    AND sst.year = cb.year
    AND sst.month = cb.month
)
SELECT
  h3_res4, year, month, latitude, longitude,
  sea_surface_temperature, chlorophyll_concentration,
  mei_value, enso_phase,
  microplastic_concentration, microplastics_source,
  current_speed, sea_level_anomaly, mixed_layer_depth,
  salinity, ocean_temperature,
  degree_heating_week, bleaching_alert_area, bleaching_alert_category,
  habitat_stress_index
FROM ranked
WHERE rn = 1;
-- CMEMS now joins on h3_res3 instead of h3_res4
-- Resolution 3 cells large enough for both grids to overlap
-- All other joins unchanged


In [0]:
%sql
SELECT
  count(*) AS total_rows,
  count(CASE WHEN sea_surface_temperature IS NULL THEN 1 END) AS null_sst,
  count(CASE WHEN chlorophyll_concentration IS NOT NULL THEN 1 END) AS rows_with_cc,
  count(CASE WHEN enso_phase IS NOT NULL THEN 1 END) AS rows_with_enso,
  count(CASE WHEN current_speed IS NOT NULL THEN 1 END) AS rows_with_cmems,
  count(CASE WHEN degree_heating_week IS NOT NULL THEN 1 END) AS rows_with_bleaching,
  count(CASE WHEN microplastic_concentration IS NOT NULL THEN 1 END) AS rows_with_mp,
  count(DISTINCT habitat_stress_index) AS stress_index_values
FROM mcphersonsharyn_silver.environmental_conditions_silver;
-- Key check: rows_with_cmems should now be much higher than 20,400


In [0]:
%sql
-- KNOWN LIMITATION — CMEMS JOIN
-- SST source files use integer degree coordinates (1 degree spacing)
-- CMEMS uses 0.083 degree resolution coordinates
-- H3 spatial join at any resolution produces sparse overlap
-- Only 22,400 of 2,795,120 SST rows have CMEMS data joined (0.8%)
-- This is a coordinate resolution mismatch not a data quality issue
-- CMEMS variables (current_speed, salinity, MLD) are available
-- for the 22,400 matched rows and are valid where present
-- Full CMEMS integration would require nearest-neighbour coordinate
-- matching which is planned for ATLAS Phase 6
-- Core ATLAS finding (ENSO SST inversion) confirmed without CMEMS

In [0]:
%sql
SELECT 'sst_silver' AS table_name, count(*) AS rows FROM mcphersonsharyn_silver.sst_silver
UNION ALL
SELECT 'cc_silver', count(*) FROM mcphersonsharyn_silver.cc_silver
UNION ALL
SELECT 'enso_silver', count(*) FROM mcphersonsharyn_silver.enso_silver
UNION ALL
SELECT 'environmental_conditions_silver', count(*) FROM mcphersonsharyn_silver.environmental_conditions_silver;
-- Morning check confirms Silver tables still exist
-- Databricks free tier persists data between sessions

In [0]:
%sql
SELECT DISTINCT
  latitude,
  longitude
FROM mcphersonsharyn_silver.sst_silver
WHERE year = 2009
AND month = 6
ORDER BY latitude, longitude
LIMIT 20;
-- Shows exact SST coordinate values
-- Confirms whether SST uses integer degrees or 0.25 degree grid
-- This determines the rounding strategy for nearest neighbour join

In [0]:
%sql
SELECT DISTINCT
  latitude,
  longitude
FROM mcphersonsharyn_silver.cmems_silver
WHERE year = 2009
AND month = 6
ORDER BY latitude, longitude
LIMIT 20;
-- Shows exact CMEMS coordinate values
-- Confirms 0.083 degree spacing
-- We need to round CMEMS coordinates to nearest integer
-- to match SST coordinates for the nearest neighbour join

In [0]:
%sql
-- Test nearest neighbour join for one month
SELECT
  count(*) AS joined_rows
FROM mcphersonsharyn_silver.sst_silver sst
JOIN (
  SELECT
    ROUND(latitude) AS lat_rounded,
    ROUND(longitude) AS lon_rounded,
    year,
    month,
    AVG(current_speed) AS current_speed,
    AVG(sea_level_anomaly) AS sea_level_anomaly,
    AVG(mixed_layer_depth) AS mixed_layer_depth,
    AVG(salinity) AS salinity,
    AVG(ocean_temperature) AS ocean_temperature
  FROM mcphersonsharyn_silver.cmems_silver
  GROUP BY ROUND(latitude), ROUND(longitude), year, month
) cmems_rounded
  ON sst.latitude = cmems_rounded.lat_rounded
  AND sst.longitude = cmems_rounded.lon_rounded
  AND sst.year = cmems_rounded.year
  AND sst.month = cmems_rounded.month
WHERE sst.year = 2009
AND sst.month = 6;
-- Rounds CMEMS coordinates to nearest integer
-- Averages multiple CMEMS readings per integer degree cell
-- Tests if this gives full coverage for one month
-- Expected: should be close to total SST rows for that month

In [0]:
%sql
SELECT count(*) AS sst_rows
FROM mcphersonsharyn_silver.sst_silver
WHERE year = 2009
AND month = 6;
-- Total SST rows for June 2009
-- Compare to 5,151 CMEMS joined rows
-- Shows what percentage of SST rows now have CMEMS data

In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_silver.cmems_silver_rounded;

CREATE TABLE mcphersonsharyn_silver.cmems_silver_rounded
(
  latitude_rounded INT,
  longitude_rounded INT,
  year INT,
  month INT,
  current_speed DOUBLE,
  sea_level_anomaly DOUBLE,
  mixed_layer_depth DOUBLE,
  salinity DOUBLE,
  ocean_temperature DOUBLE
)
USING DELTA;
-- Creates CMEMS table aggregated to integer degree coordinates
-- Matches SST coordinate system for nearest neighbour join
-- Multiple CMEMS readings per integer degree cell averaged together

In [0]:
%sql
INSERT INTO mcphersonsharyn_silver.cmems_silver_rounded
SELECT
  CAST(ROUND(latitude) AS INT) AS latitude_rounded,
  CAST(ROUND(longitude) AS INT) AS longitude_rounded,
  year,
  month,
  ROUND(AVG(current_speed), 4) AS current_speed,
  ROUND(AVG(sea_level_anomaly), 4) AS sea_level_anomaly,
  ROUND(AVG(mixed_layer_depth), 2) AS mixed_layer_depth,
  ROUND(AVG(salinity), 2) AS salinity,
  ROUND(AVG(ocean_temperature), 2) AS ocean_temperature
FROM mcphersonsharyn_silver.cmems_silver
GROUP BY
  CAST(ROUND(latitude) AS INT),
  CAST(ROUND(longitude) AS INT),
  year,
  month;
-- Rounds CMEMS coordinates to nearest integer degree
-- Averages all CMEMS readings within each integer degree cell
-- One row per integer degree cell per year per month
-- This will take several minutes given 322 million source rows

In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_silver.environmental_conditions_silver;

CREATE TABLE mcphersonsharyn_silver.environmental_conditions_silver
(
  h3_res4 STRING,
  year INT,
  month INT,
  latitude DOUBLE,
  longitude DOUBLE,
  sea_surface_temperature DOUBLE,
  chlorophyll_concentration DOUBLE,
  mei_value DOUBLE,
  enso_phase STRING,
  microplastic_concentration DOUBLE,
  microplastics_source STRING,
  current_speed DOUBLE,
  sea_level_anomaly DOUBLE,
  mixed_layer_depth DOUBLE,
  salinity DOUBLE,
  ocean_temperature DOUBLE,
  degree_heating_week DOUBLE,
  bleaching_alert_area INT,
  bleaching_alert_category STRING,
  habitat_stress_index INT
)
USING DELTA;
-- Recreating with nearest neighbour CMEMS join
-- cmems_silver_rounded aggregated to integer degree coordinates
-- Matches SST integer degree grid exactly

In [0]:
%sql
INSERT INTO mcphersonsharyn_silver.environmental_conditions_silver
WITH ranked AS (
  SELECT
    sst.h3_res4,
    sst.year,
    sst.month,
    sst.latitude,
    sst.longitude,
    sst.sea_surface_temperature,
    cc.chlorophyll_concentration,
    enso.mei_value,
    enso.enso_phase,
    mp.microplastic_concentration,
    mp.data_source AS microplastics_source,
    cmems.current_speed,
    cmems.sea_level_anomaly,
    cmems.mixed_layer_depth,
    cmems.salinity,
    cmems.ocean_temperature,
    cb.degree_heating_week,
    cb.bleaching_alert_area,
    cb.bleaching_alert_category,
    CASE WHEN sst.sea_surface_temperature > 30 THEN 1 ELSE 0 END
    + CASE WHEN cc.chlorophyll_concentration < 0.1 THEN 1 ELSE 0 END
    + CASE WHEN mp.microplastic_concentration > 1.0 THEN 1 ELSE 0 END
    + CASE WHEN cb.bleaching_alert_area >= 2 THEN 1 ELSE 0 END
    AS habitat_stress_index,
    ROW_NUMBER() OVER (
      PARTITION BY sst.h3_res4, sst.year, sst.month
      ORDER BY sst.latitude, sst.longitude
    ) AS rn
  FROM mcphersonsharyn_silver.sst_silver sst
  LEFT JOIN mcphersonsharyn_silver.cc_silver cc
    ON sst.h3_res4 = cc.h3_res4
    AND sst.year = cc.year
    AND sst.month = cc.month
  LEFT JOIN mcphersonsharyn_silver.enso_silver enso
    ON sst.year = enso.year
    AND sst.month = enso.month
  LEFT JOIN mcphersonsharyn_silver.microplastics_h3_monthly mp
    ON sst.h3_res4 = mp.h3_res4
    AND sst.month = mp.month
  LEFT JOIN mcphersonsharyn_silver.cmems_silver_rounded cmems
    ON CAST(sst.latitude AS INT) = cmems.latitude_rounded
    AND CAST(sst.longitude AS INT) = cmems.longitude_rounded
    AND sst.year = cmems.year
    AND sst.month = cmems.month
  LEFT JOIN mcphersonsharyn_silver.coral_bleaching_silver cb
    ON sst.h3_res4 = cb.h3_res4
    AND sst.year = cb.year
    AND sst.month = cb.month
)
SELECT
  h3_res4, year, month, latitude, longitude,
  sea_surface_temperature, chlorophyll_concentration,
  mei_value, enso_phase,
  microplastic_concentration, microplastics_source,
  current_speed, sea_level_anomaly, mixed_layer_depth,
  salinity, ocean_temperature,
  degree_heating_week, bleaching_alert_area, bleaching_alert_category,
  habitat_stress_index
FROM ranked
WHERE rn = 1;
-- Nearest neighbour CMEMS join on integer degree coordinates
-- CAST(sst.latitude AS INT) matches cmems.latitude_rounded
-- Expected: 74% CMEMS coverage instead of previous 0.8%
-- This will take several minutes

In [0]:
%sql
SELECT
  count(*) AS total_rows,
  count(CASE WHEN sea_surface_temperature IS NULL THEN 1 END) AS null_sst,
  count(CASE WHEN chlorophyll_concentration IS NOT NULL THEN 1 END) AS rows_with_cc,
  count(CASE WHEN enso_phase IS NOT NULL THEN 1 END) AS rows_with_enso,
  count(CASE WHEN current_speed IS NOT NULL THEN 1 END) AS rows_with_cmems,
  count(CASE WHEN degree_heating_week IS NOT NULL THEN 1 END) AS rows_with_bleaching,
  round(count(CASE WHEN current_speed IS NOT NULL THEN 1 END) * 100.0 / count(*), 1) AS cmems_coverage_pct
FROM mcphersonsharyn_silver.environmental_conditions_silver;
-- Key check: rows_with_cmems should now be around 74% of total
-- Previous: 22,400 (0.8%)
-- Expected: approximately 2,068,000 (74%)
-- cmems_coverage_pct shows the improvement clearly

In [0]:
%sql
SELECT
  latitude_rounded,
  longitude_rounded,
  year,
  month,
  current_speed
FROM mcphersonsharyn_silver.cmems_silver_rounded
WHERE year = 2009
AND month = 6
LIMIT 10;
-- Shows actual values in the rounded CMEMS table
-- Confirms latitude_rounded and longitude_rounded are integers
-- If these match SST coordinates the join should work

In [0]:
%sql
SELECT count(*) AS test_join
FROM mcphersonsharyn_silver.sst_silver sst
JOIN mcphersonsharyn_silver.cmems_silver_rounded cmems
  ON CAST(sst.latitude AS INT) = cmems.latitude_rounded
  AND CAST(sst.longitude AS INT) = cmems.longitude_rounded
  AND sst.year = cmems.year
  AND sst.month = cmems.month
WHERE sst.year = 2009
AND sst.month = 6;
-- Direct join test between SST and rounded CMEMS
-- Should return approximately 5,151 rows as before
-- If zero the CAST is not matching correctly

In [0]:
%sql
ALTER TABLE mcphersonsharyn_silver.sst_silver
ADD COLUMN latitude_int INT;

ALTER TABLE mcphersonsharyn_silver.sst_silver
ADD COLUMN longitude_int INT;
-- Adds integer coordinate columns to SST Silver
-- These will be used for the CMEMS nearest neighbour join

In [0]:
%sql
UPDATE mcphersonsharyn_silver.sst_silver
SET
  latitude_int = CAST(latitude AS INT),
  longitude_int = CAST(longitude AS INT);
-- Populates integer coordinate columns
-- latitude_int and longitude_int match cmems_silver_rounded exactly

In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_silver.environmental_conditions_silver;

CREATE TABLE mcphersonsharyn_silver.environmental_conditions_silver
(
  h3_res4 STRING,
  year INT,
  month INT,
  latitude DOUBLE,
  longitude DOUBLE,
  sea_surface_temperature DOUBLE,
  chlorophyll_concentration DOUBLE,
  mei_value DOUBLE,
  enso_phase STRING,
  microplastic_concentration DOUBLE,
  microplastics_source STRING,
  current_speed DOUBLE,
  sea_level_anomaly DOUBLE,
  mixed_layer_depth DOUBLE,
  salinity DOUBLE,
  ocean_temperature DOUBLE,
  degree_heating_week DOUBLE,
  bleaching_alert_area INT,
  bleaching_alert_category STRING,
  habitat_stress_index INT
)
USING DELTA;
-- Recreating with pre-computed integer coordinates for CMEMS join
-- latitude_int and longitude_int already in SST Silver
-- No CAST needed in the JOIN clause

In [0]:
%sql
INSERT INTO mcphersonsharyn_silver.environmental_conditions_silver
WITH ranked AS (
  SELECT
    sst.h3_res4,
    sst.year,
    sst.month,
    sst.latitude,
    sst.longitude,
    sst.sea_surface_temperature,
    cc.chlorophyll_concentration,
    enso.mei_value,
    enso.enso_phase,
    mp.microplastic_concentration,
    mp.data_source AS microplastics_source,
    cmems.current_speed,
    cmems.sea_level_anomaly,
    cmems.mixed_layer_depth,
    cmems.salinity,
    cmems.ocean_temperature,
    cb.degree_heating_week,
    cb.bleaching_alert_area,
    cb.bleaching_alert_category,
    CASE WHEN sst.sea_surface_temperature > 30 THEN 1 ELSE 0 END
    + CASE WHEN cc.chlorophyll_concentration < 0.1 THEN 1 ELSE 0 END
    + CASE WHEN mp.microplastic_concentration > 1.0 THEN 1 ELSE 0 END
    + CASE WHEN cb.bleaching_alert_area >= 2 THEN 1 ELSE 0 END
    AS habitat_stress_index,
    ROW_NUMBER() OVER (
      PARTITION BY sst.h3_res4, sst.year, sst.month
      ORDER BY sst.latitude, sst.longitude
    ) AS rn
  FROM mcphersonsharyn_silver.sst_silver sst
  LEFT JOIN mcphersonsharyn_silver.cc_silver cc
    ON sst.h3_res4 = cc.h3_res4
    AND sst.year = cc.year
    AND sst.month = cc.month
  LEFT JOIN mcphersonsharyn_silver.enso_silver enso
    ON sst.year = enso.year
    AND sst.month = enso.month
  LEFT JOIN mcphersonsharyn_silver.microplastics_h3_monthly mp
    ON sst.h3_res4 = mp.h3_res4
    AND sst.month = mp.month
  LEFT JOIN mcphersonsharyn_silver.cmems_silver_rounded cmems
    ON sst.latitude_int = cmems.latitude_rounded
    AND sst.longitude_int = cmems.longitude_rounded
    AND sst.year = cmems.year
    AND sst.month = cmems.month
  LEFT JOIN mcphersonsharyn_silver.coral_bleaching_silver cb
    ON sst.h3_res4 = cb.h3_res4
    AND sst.year = cb.year
    AND sst.month = cb.month
)
SELECT
  h3_res4, year, month, latitude, longitude,
  sea_surface_temperature, chlorophyll_concentration,
  mei_value, enso_phase,
  microplastic_concentration, microplastics_source,
  current_speed, sea_level_anomaly, mixed_layer_depth,
  salinity, ocean_temperature,
  degree_heating_week, bleaching_alert_area, bleaching_alert_category,
  habitat_stress_index
FROM ranked
WHERE rn = 1;
-- Uses pre-computed latitude_int and longitude_int from SST Silver
-- No CAST in JOIN clause — direct integer column match


In [0]:
%sql
SELECT
  count(*) AS total_rows,
  count(CASE WHEN current_speed IS NOT NULL THEN 1 END) AS rows_with_cmems,
  round(count(CASE WHEN current_speed IS NOT NULL THEN 1 END) * 100.0 / count(*), 1) AS cmems_coverage_pct,
  count(CASE WHEN chlorophyll_concentration IS NOT NULL THEN 1 END) AS rows_with_cc,
  count(CASE WHEN enso_phase IS NOT NULL THEN 1 END) AS rows_with_enso,
  count(CASE WHEN degree_heating_week IS NOT NULL THEN 1 END) AS rows_with_bleaching
FROM mcphersonsharyn_silver.environmental_conditions_silver;
-- Critical check — has CMEMS coverage improved from 0.7%?
-- Expected: approximately 74% coverage
-- rows_with_cmems should be around 2,068,000


In [0]:
%sql
-- Check what latitude_int values exist in SST Silver
SELECT DISTINCT latitude_int, longitude_int
FROM mcphersonsharyn_silver.sst_silver
WHERE year = 2009 AND month = 6
LIMIT 10;
-- Confirms latitude_int and longitude_int populated correctly
-- Should show integer values matching CMEMS rounded coordinates

In [0]:
%sql
SELECT DISTINCT latitude_rounded, longitude_rounded
FROM mcphersonsharyn_silver.cmems_silver_rounded
WHERE year = 2009 AND month = 6
AND latitude_rounded = 22
AND longitude_rounded BETWEEN 126 AND 135;
-- Checks if CMEMS rounded table has rows matching SST coordinates
-- latitude 22, longitude 126-135 should exist if join works

In [0]:
%sql
SELECT
  min(longitude_rounded) AS min_lon,
  max(longitude_rounded) AS max_lon,
  min(latitude_rounded) AS min_lat,
  max(latitude_rounded) AS max_lat,
  count(DISTINCT longitude_rounded) AS distinct_lons,
  count(DISTINCT latitude_rounded) AS distinct_lats
FROM mcphersonsharyn_silver.cmems_silver_rounded
WHERE year = 2009 AND month = 6;
-- Shows the actual geographic coverage of CMEMS rounded table
-- If longitude range is limited that explains the sparse join

In [0]:
%sql
SELECT count(*) AS matching_rows
FROM mcphersonsharyn_silver.sst_silver sst
JOIN mcphersonsharyn_silver.cmems_silver_rounded cmems
  ON sst.latitude_int = cmems.latitude_rounded
  AND sst.longitude_int = cmems.longitude_rounded
  AND sst.year = cmems.year
  AND sst.month = cmems.month
WHERE sst.year = 2009
AND sst.month = 6;
-- Direct test using pre-computed integer columns
-- Should return 5,151 rows as confirmed earlier
-- If zero the column types do not match despite looking correct

In [0]:

%sql
SELECT
  typeof(latitude_int) AS sst_lat_type,
  typeof(longitude_int) AS sst_lon_type,
  latitude_int,
  longitude_int
FROM mcphersonsharyn_silver.sst_silver
WHERE year = 2009 AND month = 6
LIMIT 3;

In [0]:
%sql
SELECT
  typeof(latitude_rounded) AS cmems_lat_type,
  typeof(longitude_rounded) AS cmems_lon_type,
  latitude_rounded,
  longitude_rounded
FROM mcphersonsharyn_silver.cmems_silver_rounded
WHERE year = 2009 AND month = 6
LIMIT 3;
-- Checks exact data types of join columns
-- Type mismatch between INT and BIGINT would cause sparse join

In [0]:
%sql
SELECT
  typeof(latitude_int) AS sst_lat_type,
  typeof(longitude_int) AS sst_lon_type,
  latitude_int,
  longitude_int
FROM mcphersonsharyn_silver.sst_silver
WHERE year = 2009 AND month = 6
LIMIT 3;

In [0]:
%sql
SELECT count(*) AS cmems_rows_in_bbox
FROM mcphersonsharyn_silver.cmems_silver_rounded
WHERE year = 2009
AND month = 6
AND latitude_rounded BETWEEN -23 AND 22
AND longitude_rounded BETWEEN -73 AND 126;
-- Counts how many CMEMS rounded rows exist within our bounding box
-- For June 2009 should be 181 lons x 46 lats = 8,326 rows

In [0]:
%sql
SELECT count(*) AS matching
FROM mcphersonsharyn_silver.sst_silver sst
JOIN mcphersonsharyn_silver.cmems_silver_rounded cmems
  ON sst.latitude_int = cmems.latitude_rounded
  AND sst.longitude_int = cmems.longitude_rounded
  AND sst.year = cmems.year
  AND sst.month = cmems.month
WHERE sst.year = 2009
AND sst.month = 6
AND sst.longitude BETWEEN -73 AND 126;
-- Adds longitude bounding box filter to SST
-- SST covers full globe -179 to 180
-- Most SST rows are outside our bounding box
-- This tests how many are within -73 to 126


In [0]:
%sql
SELECT
  count(*) AS sst_rows_in_bbox,
  count(DISTINCT longitude_int) AS distinct_lons,
  min(longitude_int) AS min_lon,
  max(longitude_int) AS max_lon
FROM mcphersonsharyn_silver.sst_silver
WHERE year = 2009
AND month = 6
AND longitude BETWEEN -73 AND 126;
-- Shows how many SST rows fall within our bounding box
-- SST covers full globe so most rows are outside -73 to 126
-- This explains why only 51 rows join with CMEMS

In [0]:
%sql
SELECT DISTINCT longitude_int
FROM mcphersonsharyn_silver.sst_silver
WHERE year = 2009
AND month = 6
AND longitude_int BETWEEN -73 AND 126
ORDER BY longitude_int;
-- Shows all SST longitude values within bounding box
-- Expected: -73, -72, -71 ... 125, 126
-- If only 2 values exist the SST grid is coarser than expected

In [0]:
%sql
SELECT
  CASE
    WHEN longitude BETWEEN -179 AND -90 THEN '-179 to -90'
    WHEN longitude BETWEEN -90 AND 0 THEN '-90 to 0'
    WHEN longitude BETWEEN 0 AND 90 THEN '0 to 90'
    WHEN longitude BETWEEN 90 AND 180 THEN '90 to 180'
  END AS lon_band,
  count(*) AS rows,
  count(DISTINCT longitude) AS distinct_lons
FROM mcphersonsharyn_silver.sst_silver
WHERE year = 2009 AND month = 6
GROUP BY lon_band
ORDER BY lon_band;
-- Shows longitude distribution across the full globe
-- Confirms where the bulk of SST data actually sits
-- The bounding box filter is working differently than expected

In [0]:
%sql
SELECT
  count(*) AS total_rows,
  count(DISTINCT longitude) AS distinct_lons,
  count(DISTINCT latitude) AS distinct_lats
FROM mcphersonsharyn_silver.sst_silver
WHERE longitude BETWEEN -73 AND 126
AND latitude BETWEEN -23 AND 22;
-- Shows how many SST rows fall within ATLAS bounding box
-- These are the rows that should join with CMEMS
-- All other rows are outside our study region

In [0]:
%sql
SELECT DISTINCT longitude
FROM mcphersonsharyn_bronze.bronze_sst_master
WHERE year = 2009
AND month = 6
AND longitude BETWEEN -73 AND 126
ORDER BY longitude
LIMIT 20;
-- Shows exact longitude values in Bronze SST within bounding box
-- Reveals the actual grid spacing of the source files

In [0]:
%sql
SELECT DISTINCT longitude
FROM mcphersonsharyn_bronze.bronze_sst_master
WHERE year = 2009
AND month = 6
ORDER BY longitude
LIMIT 30;
-- Shows the first 30 longitude values in Bronze SST
-- Reveals the actual grid pattern of the downloaded files

In [0]:
%sql
SELECT DISTINCT longitude
FROM mcphersonsharyn_bronze.bronze_sst_master
WHERE year = 2009
AND month = 6
AND longitude BETWEEN -80 AND -65
ORDER BY longitude;
-- Checks Bronze SST around -73 longitude
-- Confirms -73 exists and neighbouring values exist
-- If gap exists around -73 the filter boundary is the issue

In [0]:
%sql
SELECT DISTINCT longitude
FROM mcphersonsharyn_silver.sst_silver
WHERE year = 2009
AND month = 6
AND longitude BETWEEN -80 AND 130
ORDER BY longitude
LIMIT 30;
-- Checks what longitude values exist in SST Silver
-- around our bounding box boundaries
-- If Silver has -80 to 130 the issue is somewhere else

In [0]:
%sql
SELECT
  min(longitude) AS min_lon,
  max(longitude) AS max_lon,
  count(DISTINCT longitude) AS distinct_lons
FROM mcphersonsharyn_bronze.bronze_sst_master
WHERE year = 2009 AND month = 6;
-- Shows full longitude range in Bronze SST
-- Confirms whether the gap exists in the source data

In [0]:
%sql
SELECT DISTINCT longitude
FROM mcphersonsharyn_bronze.bronze_sst_master
WHERE year = 2009
AND month = 6
AND longitude BETWEEN -73 AND 126
ORDER BY longitude;
-- Shows ALL longitude values between -73 and 126 in Bronze SST
-- If only -73 and 126 appear the source data has no readings there

In [0]:
%sql
SELECT
  longitude,
  count(*) AS rows
FROM mcphersonsharyn_bronze.bronze_sst_master
WHERE year = 2009
AND month = 6
AND longitude BETWEEN -100 AND 150
GROUP BY longitude
ORDER BY longitude;
-- Shows all SST longitude values in our broader region
-- Reveals the actual grid pattern between -100 and 150

In [0]:
%sql
SELECT
  year,
  month,
  min(longitude) AS min_lon,
  max(longitude) AS max_lon,
  min(latitude) AS min_lat,
  max(latitude) AS max_lat,
  count(*) AS rows
FROM mcphersonsharyn_bronze.bronze_sst_master
WHERE year = 2009
AND month = 6
GROUP BY year, month;
-- Shows the actual geographic coverage of the SST files
-- Confirms what region the NOAA DITC data covers

In [0]:
%sql
SELECT
  min(longitude) AS min_lon,
  max(longitude) AS max_lon,
  count(DISTINCT longitude) AS distinct_lons,
  count(*) AS total_rows
FROM mcphersonsharyn_silver.sst_silver
WHERE year = 2009
AND month = 6;
-- Compares Silver coverage to Bronze
-- Bronze has 7,452 rows covering -179 to 180
-- Silver should have similar coverage
-- If Silver has fewer distinct longitudes the filter removed them

In [0]:
%sql
SELECT
  longitude,
  longitude - LAG(longitude) OVER (ORDER BY longitude) AS gap
FROM (
  SELECT DISTINCT longitude
  FROM mcphersonsharyn_silver.sst_silver
  WHERE year = 2009 AND month = 6
  ORDER BY longitude
) t
WHERE longitude BETWEEN -80 AND 135
ORDER BY longitude;
-- Shows the gap between consecutive longitude values
-- Reveals actual grid spacing around our bounding box
-- If gap is not 1 degree the grid is irregular

In [0]:
%sql
LIST '/Volumes/workspace/raw/environmental_data/CoralBleachingMonthly/';
-- Confirms monthly coral bleaching file uploaded correctly
-- Should show dhw_5km_monthly_1985_2026.csv at approximately 2.9GB

In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_bronze.bronze_coral_bleaching_monthly;

CREATE TABLE mcphersonsharyn_bronze.bronze_coral_bleaching_monthly
(
  time STRING,
  latitude STRING,
  longitude STRING,
  CRW_BAA STRING,
  CRW_DHW STRING
)
USING DELTA;
-- Creates empty Bronze table for monthly coral bleaching
-- All columns stored as STRING at Bronze — cast to correct types in Silver
-- Monthly data replaces annual stride-365 version

In [0]:
%sql
INSERT INTO mcphersonsharyn_bronze.bronze_coral_bleaching_monthly
SELECT
  time,
  latitude,
  longitude,
  CRW_BAA,
  CRW_DHW
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/CoralBleachingMonthly/dhw_5km_monthly_1985_2026.csv',
  format => 'csv',
  header => true,
  inferSchema => false
)
WHERE time != 'UTC'
AND latitude != 'degrees_north';

-- INSERT INTO works reliably where CREATE TABLE AS SELECT fails
-- Filters out units row where time = UTC
-- Note: Takes longer than previous queries

In [0]:
%sql
SELECT
  'bronze_coral_bleaching_monthly' AS table_name,
  count(*) AS total_rows,
  count(DISTINCT YEAR(TO_DATE(time, "yyyy-MM-dd'T'HH:mm:ss'Z'"))) AS distinct_years,
  min(TO_DATE(time, "yyyy-MM-dd'T'HH:mm:ss'Z'")) AS earliest_date,
  max(TO_DATE(time, "yyyy-MM-dd'T'HH:mm:ss'Z'")) AS latest_date
FROM mcphersonsharyn_bronze.bronze_coral_bleaching_monthly;
-- Confirms 72 million rows loaded correctly
-- Expected: 42 distinct years 1985 to 2026
-- Dates should show monthly snapshots not annual

In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_silver.coral_bleaching_silver;

CREATE TABLE mcphersonsharyn_silver.coral_bleaching_silver
(
  latitude DOUBLE,
  longitude DOUBLE,
  observation_date DATE,
  year INT,
  month INT,
  degree_heating_week DOUBLE,
  bleaching_alert_area INT,
  bleaching_alert_category STRING,
  h3_res4 STRING
)
USING DELTA;
-- Recreating from monthly Bronze source
-- 72 million rows replacing 18 million annual snapshots
-- Monthly coverage gives much better habitat stress index

In [0]:
%sql
INSERT INTO mcphersonsharyn_silver.coral_bleaching_silver
SELECT
  TRY_CAST(latitude AS DOUBLE) AS latitude,
  TRY_CAST(longitude AS DOUBLE) AS longitude,
  TO_DATE(time, "yyyy-MM-dd'T'HH:mm:ss'Z'") AS observation_date,
  YEAR(TO_DATE(time, "yyyy-MM-dd'T'HH:mm:ss'Z'")) AS year,
  MONTH(TO_DATE(time, "yyyy-MM-dd'T'HH:mm:ss'Z'")) AS month,
  TRY_CAST(CRW_DHW AS DOUBLE) AS degree_heating_week,
  TRY_CAST(CRW_BAA AS INT) AS bleaching_alert_area,
  CASE TRY_CAST(CRW_BAA AS INT)
    WHEN 0 THEN 'No Stress'
    WHEN 1 THEN 'Watch'
    WHEN 2 THEN 'Warning'
    WHEN 3 THEN 'Alert Level 1'
    WHEN 4 THEN 'Alert Level 2'
    WHEN 5 THEN 'Alert Level 2+'
    ELSE 'Unknown'
  END AS bleaching_alert_category,
  h3_longlatash3(
    TRY_CAST(longitude AS DOUBLE),
    TRY_CAST(latitude AS DOUBLE),
    4
  ) AS h3_res4
FROM mcphersonsharyn_bronze.bronze_coral_bleaching_monthly
WHERE time != 'UTC'
AND TRY_CAST(latitude AS DOUBLE) BETWEEN -23 AND 22
AND TRY_CAST(CRW_DHW AS DOUBLE) >= 0
AND TRY_CAST(CRW_DHW AS DOUBLE) IS NOT NULL
AND CRW_DHW != 'NaN';
-- Reads from monthly Bronze table
-- Filters NaN DHW values 
-- Expected: approximately 45 million clean rows
-- Note: This query will take longer than others.

In [0]:
%sql
SELECT
  'coral_bleaching_silver' AS table_name,
  count(*) AS total_rows,
  count(DISTINCT year) AS distinct_years,
  min(year) AS earliest_year,
  max(year) AS latest_year,
  count(DISTINCT month) AS distinct_months,
  round(avg(degree_heating_week), 2) AS avg_dhw,
  round(max(degree_heating_week), 2) AS max_dhw,
  count(CASE WHEN bleaching_alert_area >= 2 THEN 1 END) AS bleaching_events,
  count(CASE WHEN h3_res4 IS NULL THEN 1 END) AS null_h3
FROM mcphersonsharyn_silver.coral_bleaching_silver;
-- Confirms monthly data loaded correctly
-- distinct_months should now be 12 not 1
-- bleaching_events should be much higher than previous 315,208
-- This confirms monthly coverage vs annual snapshots

In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_silver.environmental_conditions_silver;

CREATE TABLE mcphersonsharyn_silver.environmental_conditions_silver
(
  h3_res4 STRING,
  year INT,
  month INT,
  latitude DOUBLE,
  longitude DOUBLE,
  sea_surface_temperature DOUBLE,
  chlorophyll_concentration DOUBLE,
  mei_value DOUBLE,
  enso_phase STRING,
  microplastic_concentration DOUBLE,
  microplastics_source STRING,
  current_speed DOUBLE,
  sea_level_anomaly DOUBLE,
  mixed_layer_depth DOUBLE,
  salinity DOUBLE,
  ocean_temperature DOUBLE,
  degree_heating_week DOUBLE,
  bleaching_alert_area INT,
  bleaching_alert_category STRING,
  habitat_stress_index INT
)
USING DELTA;
-- Rebuilding with monthly coral bleaching data
-- bleaching coverage should increase significantly
-- from 16,318 rows to potentially hundreds of thousands

In [0]:
%sql
INSERT INTO mcphersonsharyn_silver.environmental_conditions_silver
WITH ranked AS (
  SELECT
    sst.h3_res4,
    sst.year,
    sst.month,
    sst.latitude,
    sst.longitude,
    sst.sea_surface_temperature,
    cc.chlorophyll_concentration,
    enso.mei_value,
    enso.enso_phase,
    mp.microplastic_concentration,
    mp.data_source AS microplastics_source,
    cmems.current_speed,
    cmems.sea_level_anomaly,
    cmems.mixed_layer_depth,
    cmems.salinity,
    cmems.ocean_temperature,
    cb.degree_heating_week,
    cb.bleaching_alert_area,
    cb.bleaching_alert_category,
    CASE WHEN sst.sea_surface_temperature > 30 THEN 1 ELSE 0 END
    + CASE WHEN cc.chlorophyll_concentration < 0.1 THEN 1 ELSE 0 END
    + CASE WHEN mp.microplastic_concentration > 1.0 THEN 1 ELSE 0 END
    + CASE WHEN cb.bleaching_alert_area >= 2 THEN 1 ELSE 0 END
    AS habitat_stress_index,
    ROW_NUMBER() OVER (
      PARTITION BY sst.h3_res4, sst.year, sst.month
      ORDER BY sst.latitude, sst.longitude
    ) AS rn
  FROM mcphersonsharyn_silver.sst_silver sst
  LEFT JOIN mcphersonsharyn_silver.cc_silver cc
    ON sst.h3_res4 = cc.h3_res4
    AND sst.year = cc.year
    AND sst.month = cc.month
  LEFT JOIN mcphersonsharyn_silver.enso_silver enso
    ON sst.year = enso.year
    AND sst.month = enso.month
  LEFT JOIN mcphersonsharyn_silver.microplastics_h3_monthly mp
    ON sst.h3_res4 = mp.h3_res4
    AND sst.month = mp.month
  LEFT JOIN mcphersonsharyn_silver.cmems_silver_rounded cmems
    ON sst.latitude_int = cmems.latitude_rounded
    AND sst.longitude_int = cmems.longitude_rounded
    AND sst.year = cmems.year
    AND sst.month = cmems.month
  LEFT JOIN mcphersonsharyn_silver.coral_bleaching_silver cb
    ON sst.h3_res4 = cb.h3_res4
    AND sst.year = cb.year
    AND sst.month = cb.month
)
SELECT
  h3_res4, year, month, latitude, longitude,
  sea_surface_temperature, chlorophyll_concentration,
  mei_value, enso_phase,
  microplastic_concentration, microplastics_source,
  current_speed, sea_level_anomaly, mixed_layer_depth,
  salinity, ocean_temperature,
  degree_heating_week, bleaching_alert_area, bleaching_alert_category,
  habitat_stress_index
FROM ranked
WHERE rn = 1;
-- Rebuilt with monthly coral bleaching data
-- bleaching coverage expected to increase significantly
-- All other joins unchanged from previous version

In [0]:
%sql
SELECT
  count(*) AS total_rows,
  count(CASE WHEN sea_surface_temperature IS NULL THEN 1 END) AS null_sst,
  count(CASE WHEN chlorophyll_concentration IS NOT NULL THEN 1 END) AS rows_with_cc,
  count(CASE WHEN enso_phase IS NOT NULL THEN 1 END) AS rows_with_enso,
  count(CASE WHEN current_speed IS NOT NULL THEN 1 END) AS rows_with_cmems,
  count(CASE WHEN degree_heating_week IS NOT NULL THEN 1 END) AS rows_with_bleaching,
  round(count(CASE WHEN degree_heating_week IS NOT NULL THEN 1 END) * 100.0 / count(*), 1) AS bleaching_coverage_pct,
  count(DISTINCT habitat_stress_index) AS stress_index_values
FROM mcphersonsharyn_silver.environmental_conditions_silver;
-- Key check: rows_with_bleaching should be much higher than 16,318
-- Previous: 16,318 (0.6%)
-- Expected: significantly higher with monthly data
-- bleaching_coverage_pct shows the improvement

In [0]:
%sql
-- Compare H3 cells between SST and coral bleaching Silver
SELECT
  count(DISTINCT s.h3_res4) AS sst_h3_cells,
  count(DISTINCT c.h3_res4) AS bleaching_h3_cells
FROM mcphersonsharyn_silver.sst_silver s
FULL OUTER JOIN mcphersonsharyn_silver.coral_bleaching_silver c
  ON s.h3_res4 = c.h3_res4;
-- Checks how many H3 cells overlap between SST and bleaching
-- If very few overlap the coordinate resolution mismatch is the cause

In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_silver.coral_bleaching_silver_rounded;

CREATE TABLE mcphersonsharyn_silver.coral_bleaching_silver_rounded
(
  latitude_rounded INT,
  longitude_rounded INT,
  year INT,
  month INT,
  degree_heating_week DOUBLE,
  bleaching_alert_area INT,
  bleaching_alert_category STRING
)
USING DELTA;
-- Aggregates coral bleaching to integer degree coordinates
-- Matches SST integer degree grid for nearest neighbour join
-- Same approach used successfully for CMEMS

In [0]:
%sql
INSERT INTO mcphersonsharyn_silver.coral_bleaching_silver_rounded
SELECT
  CAST(ROUND(latitude) AS INT) AS latitude_rounded,
  CAST(ROUND(longitude) AS INT) AS longitude_rounded,
  year,
  month,
  ROUND(AVG(degree_heating_week), 2) AS degree_heating_week,
  CAST(MAX(bleaching_alert_area) AS INT) AS bleaching_alert_area,
  MAX(bleaching_alert_category) AS bleaching_alert_category
FROM mcphersonsharyn_silver.coral_bleaching_silver
GROUP BY
  CAST(ROUND(latitude) AS INT),
  CAST(ROUND(longitude) AS INT),
  year,
  month;
-- Rounds coordinates to nearest integer degree
-- Uses MAX for bleaching_alert_area — highest alert in each cell
-- Uses AVG for DHW — average heat stress in each cell
-- This will take several minutes given 45 million source rows

In [0]:
 %sql
DROP TABLE IF EXISTS mcphersonsharyn_silver.environmental_conditions_silver;

CREATE TABLE mcphersonsharyn_silver.environmental_conditions_silver
(
  h3_res4 STRING,
  year INT,
  month INT,
  latitude DOUBLE,
  longitude DOUBLE,
  sea_surface_temperature DOUBLE,
  chlorophyll_concentration DOUBLE,
  mei_value DOUBLE,
  enso_phase STRING,
  microplastic_concentration DOUBLE,
  microplastics_source STRING,
  current_speed DOUBLE,
  sea_level_anomaly DOUBLE,
  mixed_layer_depth DOUBLE,
  salinity DOUBLE,
  ocean_temperature DOUBLE,
  degree_heating_week DOUBLE,
  bleaching_alert_area INT,
  bleaching_alert_category STRING,
  habitat_stress_index INT
)
USING DELTA;
-- Rebuilding with nearest neighbour join for both CMEMS and coral bleaching
-- Both rounded to integer degree coordinates matching SST grid

In [0]:
%sql
INSERT INTO mcphersonsharyn_silver.environmental_conditions_silver
WITH ranked AS (
  SELECT
    sst.h3_res4,
    sst.year,
    sst.month,
    sst.latitude,
    sst.longitude,
    sst.sea_surface_temperature,
    cc.chlorophyll_concentration,
    enso.mei_value,
    enso.enso_phase,
    mp.microplastic_concentration,
    mp.data_source AS microplastics_source,
    cmems.current_speed,
    cmems.sea_level_anomaly,
    cmems.mixed_layer_depth,
    cmems.salinity,
    cmems.ocean_temperature,
    cb.degree_heating_week,
    cb.bleaching_alert_area,
    cb.bleaching_alert_category,
    CASE WHEN sst.sea_surface_temperature > 30 THEN 1 ELSE 0 END
    + CASE WHEN cc.chlorophyll_concentration < 0.1 THEN 1 ELSE 0 END
    + CASE WHEN mp.microplastic_concentration > 1.0 THEN 1 ELSE 0 END
    + CASE WHEN cb.bleaching_alert_area >= 2 THEN 1 ELSE 0 END
    AS habitat_stress_index,
    ROW_NUMBER() OVER (
      PARTITION BY sst.h3_res4, sst.year, sst.month
      ORDER BY sst.latitude, sst.longitude
    ) AS rn
  FROM mcphersonsharyn_silver.sst_silver sst
  LEFT JOIN mcphersonsharyn_silver.cc_silver cc
    ON sst.h3_res4 = cc.h3_res4
    AND sst.year = cc.year
    AND sst.month = cc.month
  LEFT JOIN mcphersonsharyn_silver.enso_silver enso
    ON sst.year = enso.year
    AND sst.month = enso.month
  LEFT JOIN mcphersonsharyn_silver.microplastics_h3_monthly mp
    ON sst.h3_res4 = mp.h3_res4
    AND sst.month = mp.month
  LEFT JOIN mcphersonsharyn_silver.cmems_silver_rounded cmems
    ON sst.latitude_int = cmems.latitude_rounded
    AND sst.longitude_int = cmems.longitude_rounded
    AND sst.year = cmems.year
    AND sst.month = cmems.month
  LEFT JOIN mcphersonsharyn_silver.coral_bleaching_silver_rounded cb
    ON sst.latitude_int = cb.latitude_rounded
    AND sst.longitude_int = cb.longitude_rounded
    AND sst.year = cb.year
    AND sst.month = cb.month
)
SELECT
  h3_res4, year, month, latitude, longitude,
  sea_surface_temperature, chlorophyll_concentration,
  mei_value, enso_phase,
  microplastic_concentration, microplastics_source,
  current_speed, sea_level_anomaly, mixed_layer_depth,
  salinity, ocean_temperature,
  degree_heating_week, bleaching_alert_area, bleaching_alert_category,
  habitat_stress_index
FROM ranked
WHERE rn = 1;
-- Fixed typo: degree_heating_week not degree_hunting_week
-- CMEMS and coral bleaching both joining on integer degree coordinates

In [0]:
%sql
SELECT
  count(*) AS total_rows,
  count(CASE WHEN sea_surface_temperature IS NULL THEN 1 END) AS null_sst,
  count(CASE WHEN chlorophyll_concentration IS NOT NULL THEN 1 END) AS rows_with_cc,
  count(CASE WHEN enso_phase IS NOT NULL THEN 1 END) AS rows_with_enso,
  count(CASE WHEN current_speed IS NOT NULL THEN 1 END) AS rows_with_cmems,
  count(CASE WHEN degree_heating_week IS NOT NULL THEN 1 END) AS rows_with_bleaching,
  round(count(CASE WHEN current_speed IS NOT NULL THEN 1 END) * 100.0 / count(*), 1) AS cmems_pct,
  round(count(CASE WHEN degree_heating_week IS NOT NULL THEN 1 END) * 100.0 / count(*), 1) AS bleaching_pct,
  count(DISTINCT habitat_stress_index) AS stress_index_values
FROM mcphersonsharyn_silver.environmental_conditions_silver;
-- Key checks:
-- bleaching_pct should now be significantly higher than 0.7%
-- cmems_pct should still be around 0.7%
-- stress_index_values should be 4 or 5

In [0]:
%sql
SELECT count(*) AS test_join
FROM mcphersonsharyn_silver.sst_silver sst
JOIN mcphersonsharyn_silver.coral_bleaching_silver_rounded cb
  ON sst.latitude_int = cb.latitude_rounded
  AND sst.longitude_int = cb.longitude_rounded
  AND sst.year = cb.year
  AND sst.month = cb.month
WHERE sst.year = 2009
AND sst.month = 6;
-- Tests direct join between SST and rounded bleaching table
-- If this returns few rows the coordinate ranges do not overlap
-- Expected: should return thousands of rows

In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_silver.sst_silver;

CREATE TABLE mcphersonsharyn_silver.sst_silver
(
  latitude DOUBLE,
  longitude DOUBLE,
  latitude_int INT,
  longitude_int INT,
  sea_surface_temperature DOUBLE,
  year INT,
  month INT,
  h3_res4 STRING,
  h3_res5 STRING
)
USING DELTA;
-- Recreating SST Silver with Gulf of Mexico data
-- latitude_int and longitude_int for CMEMS nearest neighbour join

In [0]:
%sql
INSERT INTO mcphersonsharyn_silver.sst_silver
SELECT
  CAST(latitude AS DOUBLE) AS latitude,
  CAST(longitude AS DOUBLE) AS longitude,
  CAST(ROUND(latitude) AS INT) AS latitude_int,
  CAST(ROUND(longitude) AS INT) AS longitude_int,
  CAST(sst_raw AS DOUBLE) AS sea_surface_temperature,
  year,
  month,
  h3_longlatash3(CAST(longitude AS DOUBLE), CAST(latitude AS DOUBLE), 4) AS h3_res4,
  h3_longlatash3(CAST(longitude AS DOUBLE), CAST(latitude AS DOUBLE), 5) AS h3_res5
FROM mcphersonsharyn_bronze.bronze_sst_master
WHERE CAST(sst_raw AS DOUBLE) != -999
AND CAST(sst_raw AS DOUBLE) IS NOT NULL
AND CAST(sst_raw AS DOUBLE) BETWEEN -2 AND 40
AND CAST(latitude AS DOUBLE) BETWEEN -23 AND 22;
-- Rebuilds SST Silver including new Gulf of Mexico rows
-- No longitude filter — keeps full global coverage
-- latitude_int and longitude_int calculated using ROUND()
-- Expected: approximately 4.2 million rows

In [0]:
%sql
SELECT
  count(*) AS total_rows,
  count(DISTINCT year) AS distinct_years,
  min(longitude) AS min_lon,
  max(longitude) AS max_lon,
  count(DISTINCT longitude) AS distinct_lons,
  round(min(sea_surface_temperature), 2) AS min_sst,
  round(max(sea_surface_temperature), 2) AS max_sst
FROM mcphersonsharyn_silver.sst_silver
WHERE year = 2009
AND month = 6;
-- Confirms Gulf of Mexico longitudes now present
-- distinct_lons should be higher than before
-- longitude range should now include -97 to -72

In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_silver.environmental_conditions_silver;

CREATE TABLE mcphersonsharyn_silver.environmental_conditions_silver
(
  h3_res4 STRING,
  year INT,
  month INT,
  latitude DOUBLE,
  longitude DOUBLE,
  sea_surface_temperature DOUBLE,
  chlorophyll_concentration DOUBLE,
  mei_value DOUBLE,
  enso_phase STRING,
  microplastic_concentration DOUBLE,
  microplastics_source STRING,
  current_speed DOUBLE,
  sea_level_anomaly DOUBLE,
  mixed_layer_depth DOUBLE,
  salinity DOUBLE,
  ocean_temperature DOUBLE,
  degree_heating_week DOUBLE,
  bleaching_alert_area INT,
  bleaching_alert_category STRING,
  habitat_stress_index INT
)
USING DELTA;
-- Rebuilding with Gulf of Mexico SST now included
-- All joins unchanged — more SST rows means better coverage

In [0]:
%sql
INSERT INTO mcphersonsharyn_silver.environmental_conditions_silver
WITH ranked AS (
  SELECT
    sst.h3_res4,
    sst.year,
    sst.month,
    sst.latitude,
    sst.longitude,
    sst.sea_surface_temperature,
    cc.chlorophyll_concentration,
    enso.mei_value,
    enso.enso_phase,
    mp.microplastic_concentration,
    mp.data_source AS microplastics_source,
    cmems.current_speed,
    cmems.sea_level_anomaly,
    cmems.mixed_layer_depth,
    cmems.salinity,
    cmems.ocean_temperature,
    cb.degree_heating_week,
    cb.bleaching_alert_area,
    cb.bleaching_alert_category,
    CASE WHEN sst.sea_surface_temperature > 30 THEN 1 ELSE 0 END
    + CASE WHEN cc.chlorophyll_concentration < 0.1 THEN 1 ELSE 0 END
    + CASE WHEN mp.microplastic_concentration > 1.0 THEN 1 ELSE 0 END
    + CASE WHEN cb.bleaching_alert_area >= 2 THEN 1 ELSE 0 END
    AS habitat_stress_index,
    ROW_NUMBER() OVER (
      PARTITION BY sst.h3_res4, sst.year, sst.month
      ORDER BY sst.latitude, sst.longitude
    ) AS rn
  FROM mcphersonsharyn_silver.sst_silver sst
  LEFT JOIN mcphersonsharyn_silver.cc_silver cc
    ON sst.h3_res4 = cc.h3_res4
    AND sst.year = cc.year
    AND sst.month = cc.month
  LEFT JOIN mcphersonsharyn_silver.enso_silver enso
    ON sst.year = enso.year
    AND sst.month = enso.month
  LEFT JOIN mcphersonsharyn_silver.microplastics_h3_monthly mp
    ON sst.h3_res4 = mp.h3_res4
    AND sst.month = mp.month
  LEFT JOIN mcphersonsharyn_silver.cmems_silver_rounded cmems
    ON sst.latitude_int = cmems.latitude_rounded
    AND sst.longitude_int = cmems.longitude_rounded
    AND sst.year = cmems.year
    AND sst.month = cmems.month
  LEFT JOIN mcphersonsharyn_silver.coral_bleaching_silver_rounded cb
    ON sst.latitude_int = cb.latitude_rounded
    AND sst.longitude_int = cb.longitude_rounded
    AND sst.year = cb.year
    AND sst.month = cb.month
)
SELECT
  h3_res4, year, month, latitude, longitude,
  sea_surface_temperature, chlorophyll_concentration,
  mei_value, enso_phase,
  microplastic_concentration, microplastics_source,
  current_speed, sea_level_anomaly, mixed_layer_depth,
  salinity, ocean_temperature,
  degree_heating_week, bleaching_alert_area, bleaching_alert_category,
  habitat_stress_index
FROM ranked
WHERE rn = 1;
-- Rebuilt with Gulf of Mexico SST coverage
-- More SST rows means better CMEMS and bleaching joins
-- Expected: approximately 3.2 million rows

In [0]:
%sql
SELECT
  count(*) AS total_rows,
  count(CASE WHEN current_speed IS NOT NULL THEN 1 END) AS rows_with_cmems,
  round(count(CASE WHEN current_speed IS NOT NULL THEN 1 END) * 100.0 / count(*), 1) AS cmems_pct,
  count(CASE WHEN degree_heating_week IS NOT NULL THEN 1 END) AS rows_with_bleaching,
  round(count(CASE WHEN degree_heating_week IS NOT NULL THEN 1 END) * 100.0 / count(*), 1) AS bleaching_pct,
  count(CASE WHEN chlorophyll_concentration IS NOT NULL THEN 1 END) AS rows_with_cc,
  count(DISTINCT habitat_stress_index) AS stress_index_values
FROM mcphersonsharyn_silver.environmental_conditions_silver;
-- Checks if CMEMS and bleaching coverage improved
-- Gulf of Mexico SST adds new H3 cells that may match CMEMS and bleaching

In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_silver.sst_silver;

CREATE TABLE mcphersonsharyn_silver.sst_silver
(
  latitude DOUBLE,
  longitude DOUBLE,
  latitude_int INT,
  longitude_int INT,
  sea_surface_temperature DOUBLE,
  year INT,
  month INT,
  h3_res4 STRING,
  h3_res5 STRING
)
USING DELTA;
-- Recreating SST Silver with wider latitude range
-- Extending to 31 degrees north to capture Gulf of Mexico species
-- Sperm whales at latitude 28-30, whale sharks at 19-30

In [0]:
%sql
INSERT INTO mcphersonsharyn_silver.sst_silver
SELECT
  CAST(latitude AS DOUBLE) AS latitude,
  CAST(longitude AS DOUBLE) AS longitude,
  CAST(ROUND(latitude) AS INT) AS latitude_int,
  CAST(ROUND(longitude) AS INT) AS longitude_int,
  CAST(sst_raw AS DOUBLE) AS sea_surface_temperature,
  year,
  month,
  h3_longlatash3(CAST(longitude AS DOUBLE), CAST(latitude AS DOUBLE), 4) AS h3_res4,
  h3_longlatash3(CAST(longitude AS DOUBLE), CAST(latitude AS DOUBLE), 5) AS h3_res5
FROM mcphersonsharyn_bronze.bronze_sst_master
WHERE CAST(sst_raw AS DOUBLE) != -999
AND CAST(sst_raw AS DOUBLE) IS NOT NULL
AND CAST(sst_raw AS DOUBLE) BETWEEN -2 AND 40
AND CAST(latitude AS DOUBLE) BETWEEN -23 AND 31;
-- Extended latitude from 22 to 31 to capture Gulf of Mexico species
-- Sperm whales at latitude 23-30, whale sharks at 19-30
-- All other filters unchanged

In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_silver.cmems_silver_rounded;

CREATE TABLE mcphersonsharyn_silver.cmems_silver_rounded
(
  latitude_rounded INT,
  longitude_rounded INT,
  year INT,
  month INT,
  current_speed DOUBLE,
  sea_level_anomaly DOUBLE,
  mixed_layer_depth DOUBLE,
  salinity DOUBLE,
  ocean_temperature DOUBLE
)
USING DELTA;
-- Rebuilding CMEMS rounded table with extended latitude to 31
-- Captures Gulf of Mexico above latitude 22

In [0]:
%sql
INSERT INTO mcphersonsharyn_silver.cmems_silver_rounded
SELECT
  CAST(ROUND(latitude) AS INT) AS latitude_rounded,
  CAST(ROUND(longitude) AS INT) AS longitude_rounded,
  year,
  month,
  ROUND(AVG(current_speed), 4) AS current_speed,
  ROUND(AVG(sea_level_anomaly), 4) AS sea_level_anomaly,
  ROUND(AVG(mixed_layer_depth), 2) AS mixed_layer_depth,
  ROUND(AVG(salinity), 2) AS salinity,
  ROUND(AVG(ocean_temperature), 2) AS ocean_temperature
FROM mcphersonsharyn_silver.cmems_silver
WHERE latitude BETWEEN -23 AND 31
GROUP BY
  CAST(ROUND(latitude) AS INT),
  CAST(ROUND(longitude) AS INT),
  year,
  month;
-- Extended to latitude 31 to match SST Silver
-- This will take several minutes given 322 million source rows

In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_silver.coral_bleaching_silver_rounded;

CREATE TABLE mcphersonsharyn_silver.coral_bleaching_silver_rounded
(
  latitude_rounded INT,
  longitude_rounded INT,
  year INT,
  month INT,
  degree_heating_week DOUBLE,
  bleaching_alert_area INT,
  bleaching_alert_category STRING
)
USING DELTA;
-- Rebuilding bleaching rounded table with extended latitude to 31

In [0]:
%sql
INSERT INTO mcphersonsharyn_silver.coral_bleaching_silver_rounded
SELECT
  CAST(ROUND(latitude) AS INT) AS latitude_rounded,
  CAST(ROUND(longitude) AS INT) AS longitude_rounded,
  year,
  month,
  ROUND(AVG(degree_heating_week), 2) AS degree_heating_week,
  CAST(MAX(bleaching_alert_area) AS INT) AS bleaching_alert_area,
  MAX(bleaching_alert_category) AS bleaching_alert_category
FROM mcphersonsharyn_silver.coral_bleaching_silver
WHERE latitude BETWEEN -23 AND 31
GROUP BY
  CAST(ROUND(latitude) AS INT),
  CAST(ROUND(longitude) AS INT),
  year,
  month;
-- Extended to latitude 31 to match SST Silver
-- MAX bleaching_alert_area keeps highest alert per cell

In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_silver.environmental_conditions_silver;

CREATE TABLE mcphersonsharyn_silver.environmental_conditions_silver
(
  h3_res4 STRING,
  year INT,
  month INT,
  latitude DOUBLE,
  longitude DOUBLE,
  sea_surface_temperature DOUBLE,
  chlorophyll_concentration DOUBLE,
  mei_value DOUBLE,
  enso_phase STRING,
  microplastic_concentration DOUBLE,
  microplastics_source STRING,
  current_speed DOUBLE,
  sea_level_anomaly DOUBLE,
  mixed_layer_depth DOUBLE,
  salinity DOUBLE,
  ocean_temperature DOUBLE,
  degree_heating_week DOUBLE,
  bleaching_alert_area INT,
  bleaching_alert_category STRING,
  habitat_stress_index INT
)
USING DELTA;
-- Rebuilding with extended latitude to 31
-- Captures Gulf of Mexico species habitat fully

In [0]:
%sql
INSERT INTO mcphersonsharyn_silver.environmental_conditions_silver
WITH ranked AS (
  SELECT
    sst.h3_res4,
    sst.year,
    sst.month,
    sst.latitude,
    sst.longitude,
    sst.sea_surface_temperature,
    cc.chlorophyll_concentration,
    enso.mei_value,
    enso.enso_phase,
    mp.microplastic_concentration,
    mp.data_source AS microplastics_source,
    cmems.current_speed,
    cmems.sea_level_anomaly,
    cmems.mixed_layer_depth,
    cmems.salinity,
    cmems.ocean_temperature,
    cb.degree_heating_week,
    cb.bleaching_alert_area,
    cb.bleaching_alert_category,
    CASE WHEN sst.sea_surface_temperature > 30 THEN 1 ELSE 0 END
    + CASE WHEN cc.chlorophyll_concentration < 0.1 THEN 1 ELSE 0 END
    + CASE WHEN mp.microplastic_concentration > 1.0 THEN 1 ELSE 0 END
    + CASE WHEN cb.bleaching_alert_area >= 2 THEN 1 ELSE 0 END
    AS habitat_stress_index,
    ROW_NUMBER() OVER (
      PARTITION BY sst.h3_res4, sst.year, sst.month
      ORDER BY sst.latitude, sst.longitude
    ) AS rn
  FROM mcphersonsharyn_silver.sst_silver sst
  LEFT JOIN mcphersonsharyn_silver.cc_silver cc
    ON sst.h3_res4 = cc.h3_res4
    AND sst.year = cc.year
    AND sst.month = cc.month
  LEFT JOIN mcphersonsharyn_silver.enso_silver enso
    ON sst.year = enso.year
    AND sst.month = enso.month
  LEFT JOIN mcphersonsharyn_silver.microplastics_h3_monthly mp
    ON sst.h3_res4 = mp.h3_res4
    AND sst.month = mp.month
  LEFT JOIN mcphersonsharyn_silver.cmems_silver_rounded cmems
    ON sst.latitude_int = cmems.latitude_rounded
    AND sst.longitude_int = cmems.longitude_rounded
    AND sst.year = cmems.year
    AND sst.month = cmems.month
  LEFT JOIN mcphersonsharyn_silver.coral_bleaching_silver_rounded cb
    ON sst.latitude_int = cb.latitude_rounded
    AND sst.longitude_int = cb.longitude_rounded
    AND sst.year = cb.year
    AND sst.month = cb.month
)
SELECT
  h3_res4, year, month, latitude, longitude,
  sea_surface_temperature, chlorophyll_concentration,
  mei_value, enso_phase,
  microplastic_concentration, microplastics_source,
  current_speed, sea_level_anomaly, mixed_layer_depth,
  salinity, ocean_temperature,
  degree_heating_week, bleaching_alert_area, bleaching_alert_category,
  habitat_stress_index
FROM ranked
WHERE rn = 1;
-- Extended latitude to 31 captures full Gulf of Mexico
-- All joins unchanged


In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_silver.sst_silver;

CREATE TABLE mcphersonsharyn_silver.sst_silver
(
  latitude DOUBLE,
  longitude DOUBLE,
  latitude_int INT,
  longitude_int INT,
  sea_surface_temperature DOUBLE,
  year INT,
  month INT,
  h3_res4 STRING,
  h3_res5 STRING
)
USING DELTA;
-- Rebuilding SST Silver with full global coverage
-- Original DITC data + Gulf of Mexico + Indian Ocean

In [0]:
%sql
INSERT INTO mcphersonsharyn_silver.sst_silver
SELECT
  CAST(latitude AS DOUBLE) AS latitude,
  CAST(longitude AS DOUBLE) AS longitude,
  CAST(ROUND(latitude) AS INT) AS latitude_int,
  CAST(ROUND(longitude) AS INT) AS longitude_int,
  CAST(sst_raw AS DOUBLE) AS sea_surface_temperature,
  year,
  month,
  h3_longlatash3(CAST(longitude AS DOUBLE), CAST(latitude AS DOUBLE), 4) AS h3_res4,
  h3_longlatash3(CAST(longitude AS DOUBLE), CAST(latitude AS DOUBLE), 5) AS h3_res5
FROM mcphersonsharyn_bronze.bronze_sst_master
WHERE CAST(sst_raw AS DOUBLE) != -999
AND CAST(sst_raw AS DOUBLE) IS NOT NULL
AND CAST(sst_raw AS DOUBLE) BETWEEN -2 AND 40
AND CAST(latitude AS DOUBLE) BETWEEN -23 AND 31;
-- Full global SST coverage
-- Latitude extended to 31 for Gulf of Mexico species
-- Expected: approximately 30 million rows


In [0]:
%sql
DROP TABLE IF EXISTS mcphersonsharyn_silver.environmental_conditions_silver;

CREATE TABLE mcphersonsharyn_silver.environmental_conditions_silver
(
  h3_res4 STRING,
  year INT,
  month INT,
  latitude DOUBLE,
  longitude DOUBLE,
  sea_surface_temperature DOUBLE,
  chlorophyll_concentration DOUBLE,
  mei_value DOUBLE,
  enso_phase STRING,
  microplastic_concentration DOUBLE,
  microplastics_source STRING,
  current_speed DOUBLE,
  sea_level_anomaly DOUBLE,
  mixed_layer_depth DOUBLE,
  salinity DOUBLE,
  ocean_temperature DOUBLE,
  degree_heating_week DOUBLE,
  bleaching_alert_area INT,
  bleaching_alert_category STRING,
  habitat_stress_index INT
)
USING DELTA;
-- Rebuilding with full global SST coverage
-- Indian Ocean and Gulf of Mexico now fully represented

In [0]:
%sql
INSERT INTO mcphersonsharyn_silver.environmental_conditions_silver
WITH ranked AS (
  SELECT
    sst.h3_res4,
    sst.year,
    sst.month,
    sst.latitude,
    sst.longitude,
    sst.sea_surface_temperature,
    cc.chlorophyll_concentration,
    enso.mei_value,
    enso.enso_phase,
    mp.microplastic_concentration,
    mp.data_source AS microplastics_source,
    cmems.current_speed,
    cmems.sea_level_anomaly,
    cmems.mixed_layer_depth,
    cmems.salinity,
    cmems.ocean_temperature,
    cb.degree_heating_week,
    cb.bleaching_alert_area,
    cb.bleaching_alert_category,
    CASE WHEN sst.sea_surface_temperature > 30 THEN 1 ELSE 0 END
    + CASE WHEN cc.chlorophyll_concentration < 0.1 THEN 1 ELSE 0 END
    + CASE WHEN mp.microplastic_concentration > 1.0 THEN 1 ELSE 0 END
    + CASE WHEN cb.bleaching_alert_area >= 2 THEN 1 ELSE 0 END
    AS habitat_stress_index,
    ROW_NUMBER() OVER (
      PARTITION BY sst.h3_res4, sst.year, sst.month
      ORDER BY sst.latitude, sst.longitude
    ) AS rn
  FROM mcphersonsharyn_silver.sst_silver sst
  LEFT JOIN mcphersonsharyn_silver.cc_silver cc
    ON sst.h3_res4 = cc.h3_res4
    AND sst.year = cc.year
    AND sst.month = cc.month
  LEFT JOIN mcphersonsharyn_silver.enso_silver enso
    ON sst.year = enso.year
    AND sst.month = enso.month
  LEFT JOIN mcphersonsharyn_silver.microplastics_h3_monthly mp
    ON sst.h3_res4 = mp.h3_res4
    AND sst.month = mp.month
  LEFT JOIN mcphersonsharyn_silver.cmems_silver_rounded cmems
    ON sst.latitude_int = cmems.latitude_rounded
    AND sst.longitude_int = cmems.longitude_rounded
    AND sst.year = cmems.year
    AND sst.month = cmems.month
  LEFT JOIN mcphersonsharyn_silver.coral_bleaching_silver_rounded cb
    ON sst.latitude_int = cb.latitude_rounded
    AND sst.longitude_int = cb.longitude_rounded
    AND sst.year = cb.year
    AND sst.month = cb.month
)
SELECT
  h3_res4, year, month, latitude, longitude,
  sea_surface_temperature, chlorophyll_concentration,
  mei_value, enso_phase,
  microplastic_concentration, microplastics_source,
  current_speed, sea_level_anomaly, mixed_layer_depth,
  salinity, ocean_temperature,
  degree_heating_week, bleaching_alert_area, bleaching_alert_category,
  habitat_stress_index
FROM ranked
WHERE rn = 1;
-- Full global SST now drives the join
-- Indian Ocean and Gulf of Mexico fully represented



In [0]:
%sql
SELECT
  count(*) AS total_rows,
  count(CASE WHEN current_speed IS NOT NULL THEN 1 END) AS rows_with_cmems,
  round(count(CASE WHEN current_speed IS NOT NULL THEN 1 END) * 100.0 / count(*), 1) AS cmems_pct,
  count(CASE WHEN degree_heating_week IS NOT NULL THEN 1 END) AS rows_with_bleaching,
  round(count(CASE WHEN degree_heating_week IS NOT NULL THEN 1 END) * 100.0 / count(*), 1) AS bleaching_pct,
  count(CASE WHEN chlorophyll_concentration IS NOT NULL THEN 1 END) AS rows_with_cc,
  round(count(CASE WHEN chlorophyll_concentration IS NOT NULL THEN 1 END) * 100.0 / count(*), 1) AS cc_pct,
  count(DISTINCT habitat_stress_index) AS stress_index_values
FROM mcphersonsharyn_silver.environmental_conditions_silver;
-- Confirms coverage improvements across all variables
-- All percentages should be higher than before